# Machine Learning

In [1]:
# Import pandas for working with our F1 dataset
import pandas as pd

# Load the prepared data
df = pd.read_csv("f1_2023_2025_model_ready.csv")

# Confirm the dataset loaded successfully
print("Loaded laps:", len(df))
print("Total columns:", len(df.columns))

/Users/tashmikamaharaj/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loaded laps: 74508
Total columns: 34


In [2]:
# Select the lap duration as our prediction target
y = df["lap_duration"]

# Check that every target value is usable
print("Target:", y.name)
print("Missing values:", y.isna().sum())
print("Zero or negative values:", (y <= 0).sum())
print("Total observations:", len(y))

Target: lap_duration
Missing values: 0
Zero or negative values: 0
Total observations: 74508


In [3]:
# Select tyre and weather features for our first model
feature_columns = [
    "compound",
    "tyre_age",
    "air_temperature",
    "track_temperature",
    "humidity",
    "rainfall",
    "wind_speed",
    "pressure"
]

# Create our input dataset
X = df[feature_columns].copy()

# Check the selected features and missing values
print("Input features:", X.shape[1])
print("Missing values by feature:")
print(X.isna().sum())

Input features: 8
Missing values by feature:
compound              0
tyre_age              0
air_temperature      19
track_temperature    19
humidity             19
rainfall             19
wind_speed           19
pressure             19
dtype: int64


In [4]:
# Find laps with all eight input features available
complete_rows = X.notna().all(axis=1)

# Keep only complete inputs and their matching target values
X = X.loc[complete_rows].copy()
y = y.loc[complete_rows].copy()

# Confirm inputs and targets still match
print("Input rows:", len(X))
print("Target rows:", len(y))
print("Remaining missing values:", X.isna().sum().sum())

Input rows: 74489
Target rows: 74489
Remaining missing values: 0


In [5]:
# Import the tool for splitting data by race
from sklearn.model_selection import GroupShuffleSplit

# Identify the race session for each remaining lap
race_groups = df.loc[X.index, "session_key"]

# Create a reproducible 80/20 split by race
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# Get the training and testing row positions
train_idx, test_idx = next(
    splitter.split(X, y, groups=race_groups)
)

# Separate our inputs and targets
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Check that no race appears in both groups
train_races = set(race_groups.iloc[train_idx])
test_races = set(race_groups.iloc[test_idx])

print("Training laps:", len(X_train))
print("Testing laps:", len(X_test))
print("Training races:", len(train_races))
print("Testing races:", len(test_races))
print("Overlapping races:", len(train_races & test_races))

Training laps: 58814
Testing laps: 15675
Training races: 56
Testing races: 14
Overlapping races: 0


## Baseline Model

In [6]:
# Import tools for measuring prediction errors
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Train a simple baseline that always predicts the training mean
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

# Predict lap durations for unseen races
baseline_predictions = baseline.predict(X_test)

# Calculate baseline errors in seconds
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

print("Baseline MAE:", round(baseline_mae, 2), "seconds")
print("Baseline RMSE:", round(baseline_rmse, 2), "seconds")

Baseline MAE: 9.66 seconds
Baseline RMSE: 13.63 seconds


## Random Forest Regression Model

In [7]:
# Import tools for preparing features and training Random Forest
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Convert tyre compound names into numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ("compound", OneHotEncoder(handle_unknown="ignore"), ["compound"])
    ],
    remainder="passthrough"
)

# Build a pipeline that prepares features and trains the model
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using only our training races
rf_model.fit(X_train, y_train)

print("Random Forest training complete!")

Random Forest training complete!


In [8]:
# Predict lap times for our unseen test races
rf_predictions = rf_model.predict(X_test)

# Calculate the Random Forest's prediction errors
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

# Compare the Random Forest against our baseline
print("Baseline MAE:", round(baseline_mae, 2), "seconds")
print("Random Forest MAE:", round(rf_mae, 2), "seconds")

print("Baseline RMSE:", round(baseline_rmse, 2), "seconds")
print("Random Forest RMSE:", round(rf_rmse, 2), "seconds")

Baseline MAE: 9.66 seconds
Random Forest MAE: 11.9 seconds
Baseline RMSE: 13.63 seconds
Random Forest RMSE: 16.44 seconds


In [9]:
# Check which race-identifying columns are available
race_columns = [
    "race_name",
    "session_key",
    "meeting_key"
]

# Display their availability and unique values
for column in race_columns:
    print(column, "exists:", column in df.columns)

    if column in df.columns:
        print("Unique values:", df[column].nunique())

race_name exists: True
Unique values: 45
session_key exists: True
Unique values: 70
meeting_key exists: True
Unique values: 70


In [10]:
# Find race names represented in each dataset
train_names = set(df.loc[X_train.index, "race_name"])
test_names = set(df.loc[X_test.index, "race_name"])

# Identify race names appearing only in the test dataset
unseen_names = test_names - train_names

print("Training race names:", len(train_names))
print("Testing race names:", len(test_names))
print("Unseen race names:", len(unseen_names))
print("Names:", sorted(unseen_names))

Training race names: 40
Testing race names: 14
Unseen race names: 5
Names: ['Austria', 'Bahrain', 'China', 'Hungary', 'Miami']


In [11]:
# Display all race names used across the three seasons
print(sorted(df["race_name"].unique()))


['Austin', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Baku', 'Barcelona', 'Belgium', 'Brazil', 'Budapest', 'Canada', 'China', 'Hungary', 'Imola', 'Italy', 'Japan', 'Jeddah', 'Las Vegas', 'Lusail', 'Marina Bay', 'Melbourne', 'Mexico', 'Mexico City', 'Miami', 'Miami Gardens', 'Monaco', 'Montréal', 'Monza', 'Netherlands', 'Qatar', 'Sakhir', 'Saudi Arabia', 'Shanghai', 'Silverstone', 'Singapore', 'Spa-Francorchamps', 'Spain', 'Spielberg', 'Suzuka', 'São Paulo', 'United Arab Emirates', 'United Kingdom', 'United States', 'Yas Island', 'Zandvoort']


In [12]:
# Match alternative race names to consistent circuit labels
circuit_names = {
    "Austria": "Spielberg",
    "Bahrain": "Sakhir",
    "China": "Shanghai",
    "Hungary": "Budapest",
    "Miami": "Miami Gardens",
    "Belgium": "Spa-Francorchamps"
}

# Create a new column without changing the original race names
df["circuit"] = df["race_name"].replace(circuit_names)

# Check whether the number of distinct circuit labels decreased
print("Original race names:", df["race_name"].nunique())
print("Standardised circuit labels:", df["circuit"].nunique())

Original race names: 45
Standardised circuit labels: 39


In [13]:
# Standardise the remaining alternative circuit names
more_circuit_names = {
    "Australia": "Melbourne",
    "Azerbaijan": "Baku",
    "Brazil": "São Paulo",
    "Canada": "Montréal",
    "Italy": "Monza",
    "Japan": "Suzuka",
    "Mexico": "Mexico City",
    "Netherlands": "Zandvoort",
    "Qatar": "Lusail",
    "Saudi Arabia": "Jeddah",
    "Singapore": "Marina Bay",
    "Spain": "Barcelona",
    "United Arab Emirates": "Yas Island",
    "United Kingdom": "Silverstone",
    "United States": "Austin"
}

# Update our standardised circuit column
df["circuit"] = df["circuit"].replace(more_circuit_names)

# Check the remaining unique circuit labels
print("Standardised circuits:", df["circuit"].nunique())
print(sorted(df["circuit"].unique()))

Standardised circuits: 24
['Austin', 'Baku', 'Barcelona', 'Budapest', 'Imola', 'Jeddah', 'Las Vegas', 'Lusail', 'Marina Bay', 'Melbourne', 'Mexico City', 'Miami Gardens', 'Monaco', 'Montréal', 'Monza', 'Sakhir', 'Shanghai', 'Silverstone', 'Spa-Francorchamps', 'Spielberg', 'Suzuka', 'São Paulo', 'Yas Island', 'Zandvoort']


In [14]:
# Find the circuits represented in each dataset
train_circuits = set(df.loc[X_train.index, "circuit"])
test_circuits = set(df.loc[X_test.index, "circuit"])

# Identify circuits the model has never seen during training
unseen_circuits = test_circuits - train_circuits

print("Training circuits:", len(train_circuits))
print("Testing circuits:", len(test_circuits))
print("Unseen test circuits:", len(unseen_circuits))
print("Unseen names:", sorted(unseen_circuits))

Training circuits: 24
Testing circuits: 12
Unseen test circuits: 0
Unseen names: []


In [15]:
# Add circuit information to our existing features
X_with_circuit = df.loc[X.index, feature_columns + ["circuit"]].copy()

# Preserve our original race-based training and testing split
X_train_circuit = X_with_circuit.iloc[train_idx].copy()
X_test_circuit = X_with_circuit.iloc[test_idx].copy()

# Check the updated input datasets
print("Training laps:", len(X_train_circuit))
print("Testing laps:", len(X_test_circuit))
print("Input features:", X_with_circuit.shape[1])
print("Missing values:", X_with_circuit.isna().sum().sum())

Training laps: 58814
Testing laps: 15675
Input features: 9
Missing values: 0


In [16]:
# Build a new model that encodes both tyre compound and circuit
circuit_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            ["compound", "circuit"]
        )
    ],
    remainder="passthrough"
)

# Create a new Random Forest without changing our first model
rf_circuit_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using the same training races as before
rf_circuit_model.fit(X_train_circuit, y_train)

print("Circuit-aware Random Forest training complete!")

Circuit-aware Random Forest training complete!


In [17]:
# Predict lap times for the same unseen test races
rf_circuit_predictions = rf_circuit_model.predict(X_test_circuit)

# Calculate prediction errors in seconds
rf_circuit_mae = mean_absolute_error(y_test, rf_circuit_predictions)
rf_circuit_rmse = np.sqrt(
    mean_squared_error(y_test, rf_circuit_predictions)
)

# Compare all three models
print("Baseline MAE:", round(baseline_mae, 2))
print("Original Random Forest MAE:", round(rf_mae, 2))
print("Circuit-aware Random Forest MAE:", round(rf_circuit_mae, 2))

print("Baseline RMSE:", round(baseline_rmse, 2))
print("Original Random Forest RMSE:", round(rf_rmse, 2))
print("Circuit-aware Random Forest RMSE:", round(rf_circuit_rmse, 2))

Baseline MAE: 9.66
Original Random Forest MAE: 11.9
Circuit-aware Random Forest MAE: 13.35
Baseline RMSE: 13.63
Original Random Forest RMSE: 16.44
Circuit-aware Random Forest RMSE: 17.33


In [18]:
# Predict lap times for the races used during training
train_predictions = rf_circuit_model.predict(X_train_circuit)

# Calculate training and testing errors
train_mae = mean_absolute_error(y_train, train_predictions)
test_mae = mean_absolute_error(y_test, rf_circuit_predictions)

# Compare how well the model performs on familiar and unseen races
print("Training MAE:", round(train_mae, 2), "seconds")
print("Testing MAE:", round(test_mae, 2), "seconds")

Training MAE: 0.68 seconds
Testing MAE: 13.35 seconds


In [19]:
# Build a simpler Random Forest to reduce overfitting
rf_simple_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the simpler model on the same training races
rf_simple_model.fit(X_train_circuit, y_train)

# Measure training and testing errors
simple_train_mae = mean_absolute_error(
    y_train, rf_simple_model.predict(X_train_circuit)
)

simple_test_mae = mean_absolute_error(
    y_test, rf_simple_model.predict(X_test_circuit)
)

print("Simpler RF Training MAE:", round(simple_train_mae, 2))
print("Simpler RF Testing MAE:", round(simple_test_mae, 2))

Simpler RF Training MAE: 3.79
Simpler RF Testing MAE: 9.33


In [20]:
# Predict lap durations using the simpler Random Forest
simple_predictions = rf_simple_model.predict(X_test_circuit)

# Calculate RMSE and R² on the test races
from sklearn.metrics import r2_score

simple_rmse = np.sqrt(
    mean_squared_error(y_test, simple_predictions)
)

simple_r2 = r2_score(y_test, simple_predictions)

# Display the model's evaluation metrics
print("MAE:", round(simple_test_mae, 2), "seconds")
print("RMSE:", round(simple_rmse, 2), "seconds")
print("R²:", round(simple_r2, 3))

MAE: 9.33 seconds
RMSE: 13.63 seconds
R²: -0.006


The simpler model generalises better than our original Random Forest, but its predictive performance is still weak.

In [21]:
# Calculate R² for our original baseline model
baseline_r2 = r2_score(y_test, baseline_predictions)

# Compare both models using the same test races
print("Baseline R²:", round(baseline_r2, 3))
print("Simpler Random Forest R²:", round(simple_r2, 3))

Baseline R²: -0.006
Simpler Random Forest R²: -0.006


In [22]:
# Identify the race sessions in our original training data
training_groups = race_groups.iloc[train_idx]

# Create a validation split using only training races
validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=21
)

# Split the training data into development and validation sets
dev_idx, val_idx = next(
    validation_splitter.split(
        X_train_circuit,
        y_train,
        groups=training_groups
    )
)

# Check the number of races in each group
dev_races = set(training_groups.iloc[dev_idx])
val_races = set(training_groups.iloc[val_idx])

print("Development races:", len(dev_races))
print("Validation races:", len(val_races))
print("Overlapping races:", len(dev_races & val_races))

Development races: 44
Validation races: 12
Overlapping races: 0


In [23]:
# Create development inputs and targets
X_dev = X_train_circuit.iloc[dev_idx].copy()
y_dev = y_train.iloc[dev_idx].copy()

# Create validation inputs and targets
X_val = X_train_circuit.iloc[val_idx].copy()
y_val = y_train.iloc[val_idx].copy()

# Confirm the datasets have matching row counts
print("Development inputs:", len(X_dev))
print("Development targets:", len(y_dev))
print("Validation inputs:", len(X_val))
print("Validation targets:", len(y_val))

Development inputs: 46050
Development targets: 46050
Validation inputs: 12764
Validation targets: 12764


In [24]:
# Build a Random Forest for our validation experiment
rf_validation_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using development races only
rf_validation_model.fit(X_dev, y_dev)

# Predict lap times for the separate validation races
val_predictions = rf_validation_model.predict(X_val)

# Measure average prediction error
val_mae = mean_absolute_error(y_val, val_predictions)

print("Validation MAE:", round(val_mae, 2), "seconds")

Validation MAE: 13.33 seconds


In [25]:
# Train a baseline using development races only
validation_baseline = DummyRegressor(strategy="mean")
validation_baseline.fit(X_dev, y_dev)

# Predict lap times for the validation races
baseline_val_predictions = validation_baseline.predict(X_val)

# Compare baseline and Random Forest errors
baseline_val_mae = mean_absolute_error(
    y_val, baseline_val_predictions
)

print("Validation baseline MAE:", round(baseline_val_mae, 2))
print("Validation Random Forest MAE:", round(val_mae, 2))

Validation baseline MAE: 10.31
Validation Random Forest MAE: 13.33


In [26]:
# Create a table of prediction errors for each validation lap
validation_errors = pd.DataFrame({
    "race": df.loc[X_val.index, "race_name"],
    "actual": y_val,
    "predicted": val_predictions
})

# Calculate the absolute error for each lap
validation_errors["absolute_error"] = (
    validation_errors["actual"] - validation_errors["predicted"]
).abs()

# Show average prediction error for each validation race
print(
    validation_errors.groupby("race")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)

race
Baku              28.07
Brazil            26.11
Japan             19.02
United Kingdom    17.31
Suzuka            15.61
Spielberg         14.33
Canada            14.12
Lusail            12.32
Imola              6.56
Mexico             6.40
Netherlands        4.30
Monza              1.40


In [27]:
# Compare actual and predicted average lap times by race
race_predictions = (
    validation_errors.groupby("race")[["actual", "predicted"]]
    .mean()
    .round(2)
)

# Calculate prediction bias in seconds
# Positive values mean the model predicts laps that are too slow
race_predictions["bias"] = (
    race_predictions["predicted"] - race_predictions["actual"]
).round(2)

# Display the races with the largest absolute bias first
print(
    race_predictions
    .reindex(
        race_predictions["bias"].abs()
        .sort_values(ascending=False).index
    )
    .to_string()
)

                actual  predicted   bias
race                                    
Baku            109.70      81.62 -28.08
Brazil           89.76     114.09  24.33
Japan           100.03      81.02 -19.01
Suzuka           93.48     109.09  15.61
United Kingdom   95.65     111.05  15.40
Spielberg        72.49      84.85  12.36
Lusail           92.22      99.06   6.84
Canada           90.68      97.06   6.38
Mexico           86.37      80.03  -6.34
Imola            86.28      80.47  -5.81
Netherlands      76.48      80.73   4.25
Monza            87.24      86.04  -1.20


In [28]:
# Inspect circuit labels for the races with the largest errors
problem_races = ["Baku", "Brazil", "Japan", "Suzuka"]

# Show each race name alongside its standardised circuit
print(
    df[df["race_name"].isin(problem_races)]
    [["race_name", "session_key", "circuit"]]
    .drop_duplicates()
    .sort_values(["race_name", "session_key"])
    .to_string(index=False)
)

race_name  session_key   circuit
     Baku         9070      Baku
     Baku         9904      Baku
   Brazil         9636 São Paulo
    Japan         9496    Suzuka
   Suzuka         9173    Suzuka
   Suzuka        10006    Suzuka


In [29]:
# Identify the season for each lap using its timestamp
df["season"] = pd.to_datetime(
    df["date_start"], utc=True, format="mixed"
).dt.year

# Show the validation race sessions and their seasons
validation_sessions = (
    df.loc[X_val.index, ["session_key", "race_name", "circuit", "season"]]
    .drop_duplicates()
    .sort_values(["season", "race_name"])
)

print(validation_sessions.to_string(index=False))

 session_key      race_name     circuit  season
        9070           Baku        Baku  2023.0
        9221         Lusail      Lusail  2023.0
        9157          Monza       Monza  2023.0
        9636         Brazil   São Paulo  2024.0
        9531         Canada    Montréal  2024.0
        9496          Japan      Suzuka  2024.0
        9625         Mexico Mexico City  2024.0
        9582    Netherlands   Zandvoort  2024.0
        9558 United Kingdom Silverstone  2024.0
        9987          Imola       Imola  2025.0
        9955      Spielberg   Spielberg  2025.0
       10006         Suzuka      Suzuka  2025.0


In [30]:
# Count development laps for the circuits with large prediction errors
problem_circuits = ["Baku", "São Paulo", "Suzuka", "Silverstone"]

# Show how much training data the model had for each circuit
print(
    df.loc[X_dev.index]
    .query("circuit in @problem_circuits")
    .groupby("circuit")
    .agg(
        development_laps=("lap_duration", "size"),
        development_races=("session_key", "nunique")
    )
    .to_string()
)

             development_laps  development_races
circuit                                         
Baku                      947                  1
Silverstone               787                  1
Suzuka                    836                  1
São Paulo                2251                  2


In [31]:
# Calculate typical lap times at each problematic circuit
dev_pace = df.loc[X_dev.index].groupby("circuit")["lap_duration"].median()
val_pace = df.loc[X_val.index].groupby("circuit")["lap_duration"].median()

# Compare circuits that appear in both datasets
pace_comparison = pd.DataFrame({
    "Development median": dev_pace,
    "Validation median": val_pace
}).loc[["Baku", "Silverstone", "Suzuka", "São Paulo"]]

# Show the difference in typical lap times
pace_comparison["Difference"] = (
    pace_comparison["Validation median"]
    - pace_comparison["Development median"]
)

print(pace_comparison.round(2).to_string())

             Development median  Validation median  Difference
circuit                                                       
Baku                     108.98             107.34       -1.64
Silverstone              104.71              93.09      -11.61
Suzuka                    98.88              94.85       -4.03
São Paulo                 75.58              84.88        9.30


In [32]:
# Get the feature names after categorical encoding
feature_names = (
    rf_validation_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get the importance score for each feature
importance_scores = (
    rf_validation_model.named_steps["regressor"]
    .feature_importances_
)

# Display the most important features
feature_importance = pd.Series(
    importance_scores,
    index=feature_names
).sort_values(ascending=False)

print(feature_importance.head(15).round(4).to_string())

remainder__track_temperature             0.2487
remainder__air_temperature               0.1793
categories__circuit_Spa-Francorchamps    0.1686
remainder__pressure                      0.1256
remainder__tyre_age                      0.0736
categories__circuit_Austin               0.0596
remainder__humidity                      0.0401
categories__circuit_Jeddah               0.0233
categories__circuit_Yas Island           0.0232
categories__circuit_Spielberg            0.0133
categories__circuit_Baku                 0.0132
categories__circuit_Monza                0.0090
remainder__wind_speed                    0.0082
categories__circuit_Lusail               0.0060
categories__circuit_Melbourne            0.0037


In [33]:
# Find all encoded tyre compound features
compound_importance = feature_importance[
    feature_importance.index.str.contains("compound_")
]

# Show the importance of each tyre compound
print(compound_importance.round(4).to_string())

# Calculate their combined importance
print(
    "\nTotal compound importance:",
    round(compound_importance.sum(), 4)
)

categories__compound_HARD            0.0003
categories__compound_INTERMEDIATE    0.0002
categories__compound_MEDIUM          0.0001
categories__compound_SOFT            0.0000

Total compound importance: 0.0007


In [34]:
# Count how many development laps use each tyre compound
print("Development data:")
print(X_dev["compound"].value_counts().to_string())

# Count how many validation laps use each tyre compound
print("\nValidation data:")
print(X_val["compound"].value_counts().to_string())

Development data:
compound
HARD            21680
MEDIUM          17460
SOFT             5685
INTERMEDIATE     1225

Validation data:
compound
HARD            5506
MEDIUM          4658
INTERMEDIATE    2069
SOFT             503
WET               28


In [35]:
# Add tyre compounds to our existing validation error table
validation_errors["compound"] = X_val["compound"]

# Calculate average prediction error for each compound
compound_errors = (
    validation_errors.groupby("compound")["absolute_error"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

# Display the number of laps and average error
print(compound_errors.round(2).to_string())

              count   mean
compound                  
WET              28  23.63
INTERMEDIATE   2069  21.17
SOFT            503  17.69
HARD           5506  11.53
MEDIUM         4658  11.43


In [36]:
# Calculate the baseline's absolute error for each validation lap
validation_errors["baseline_error"] = (
    validation_errors["actual"].to_numpy()
    - baseline_val_predictions
).__abs__()

# Compare the Random Forest and baseline for each tyre compound
compound_comparison = (
    validation_errors.groupby("compound")
    .agg(
        laps=("absolute_error", "size"),
        rf_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_error", "mean")
    )
)

# Positive values mean the Random Forest is worse than the baseline
compound_comparison["rf_minus_baseline"] = (
    compound_comparison["rf_mae"]
    - compound_comparison["baseline_mae"]
)

print(compound_comparison.round(2).to_string())

              laps  rf_mae  baseline_mae  rf_minus_baseline
compound                                                   
HARD          5506   11.53         11.03               0.50
INTERMEDIATE  2069   21.17          8.24              12.93
MEDIUM        4658   11.43         10.31               1.12
SOFT           503   17.69         10.32               7.37
WET             28   23.63         20.64               3.00


In [37]:
# Select validation laps using intermediate tyres
intermediate_laps = validation_errors[
    validation_errors["compound"] == "INTERMEDIATE"
]

# Compare model and baseline errors for each race
intermediate_by_race = (
    intermediate_laps.groupby("race")
    .agg(
        laps=("absolute_error", "size"),
        rf_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_error", "mean")
    )
)

# Display races with the most intermediate laps first
print(
    intermediate_by_race
    .sort_values("laps", ascending=False)
    .round(2)
    .to_string()
)

                laps  rf_mae  baseline_mae
race                                      
Brazil          1085   26.22          9.19
Canada           773   15.07          5.96
United Kingdom   211   17.54         11.77


In [38]:
# Select the three races containing intermediate tyres
rainy_races = ["Brazil", "Canada", "United Kingdom"]

# Compare prediction errors by race and tyre compound
race_compound_errors = (
    validation_errors[
        validation_errors["race"].isin(rainy_races)
    ]
    .groupby(["race", "compound"])
    .agg(
        laps=("absolute_error", "size"),
        rf_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_error", "mean")
    )
)

# Display the results
print(race_compound_errors.round(2).to_string())

                             laps  rf_mae  baseline_mae
race           compound                                
Brazil         INTERMEDIATE  1085   26.22          9.19
               WET             13   16.86         39.53
Canada         HARD           107   12.20         14.16
               INTERMEDIATE   773   15.07          5.96
               MEDIUM         310   11.67         13.57
               WET             15   29.51          4.26
United Kingdom HARD            41   22.62          1.97
               INTERMEDIATE   211   17.54         11.77
               MEDIUM         500   15.11          3.14
               SOFT           162   22.44          1.55


In [39]:
# Select only circuit and tyre characteristics
tyre_features = ["circuit", "compound", "tyre_age"]

# Create development and validation inputs
X_dev_tyres = X_dev[tyre_features].copy()
X_val_tyres = X_val[tyre_features].copy()

# Encode categorical features
tyre_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            ["circuit", "compound"]
        )
    ],
    remainder="passthrough"
)

# Build a Random Forest with the same settings
rf_tyres_only = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using development races only
rf_tyres_only.fit(X_dev_tyres, y_dev)

# Evaluate on the same validation races
tyres_only_predictions = rf_tyres_only.predict(X_val_tyres)

print("Tyres-only RF MAE:", round(
    mean_absolute_error(y_val, tyres_only_predictions), 2
))

print("Original RF MAE:", round(
    mean_absolute_error(y_val, val_predictions), 2
))

print("Baseline MAE:", round(
    mean_absolute_error(y_val, baseline_val_predictions), 2
))

Tyres-only RF MAE: 9.89
Original RF MAE: 13.33
Baseline MAE: 10.31


#### Predictor Variables Used in the Tyre-and-Circuit Models

The tyre-and-circuit models used three predictor variables:

- **Circuit:** The race circuit, represented as a categorical variable and transformed using one-hot encoding.
- **Compound:** The recorded tyre compound (SOFT, MEDIUM, HARD, INTERMEDIATE or WET), also transformed using one-hot encoding.
- **Tyre age:** An estimated numerical proxy for the number of laps completed on the current tyre set.

The tyre-age variable is an estimate derived from the available lap and stint records and should not be interpreted as a directly measured indicator of physical tyre degradation.

Although weather conditions were central to the research question, the best-performing model in the five-fold comparison did not include weather predictors. Separate weather-inclusive model configurations were evaluated to investigate whether adding recorded weather measurements improved predictive accuracy.

The results therefore distinguish between the predictive contribution of tyre-and-circuit information and the observed performance of models that additionally incorporated weather variables. They do not establish the causal effects of tyre characteristics or weather conditions on lap performance.

In [40]:
# Select tyre and weather features without circuit
weather_features = feature_columns.copy()

# Use the same development and validation races
X_dev_weather = X_dev[weather_features].copy()
X_val_weather = X_val[weather_features].copy()

# Encode tyre compound
weather_preprocessor = ColumnTransformer(
    transformers=[
        ("compound", OneHotEncoder(handle_unknown="ignore"), ["compound"])
    ],
    remainder="passthrough"
)

# Build the same Random Forest without circuit
rf_weather_only = Pipeline([
    ("preprocessor", weather_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train and evaluate on the same race split
rf_weather_only.fit(X_dev_weather, y_dev)
weather_predictions = rf_weather_only.predict(X_val_weather)

print("Tyres + weather MAE:", round(
    mean_absolute_error(y_val, weather_predictions), 2
))
print("Tyres + circuit MAE:", round(
    mean_absolute_error(y_val, tyres_only_predictions), 2
))
print("All features MAE:", round(
    mean_absolute_error(y_val, val_predictions), 2
))
print("Baseline MAE:", round(
    mean_absolute_error(y_val, baseline_val_predictions), 2
))

Tyres + weather MAE: 16.21
Tyres + circuit MAE: 9.89
All features MAE: 13.33
Baseline MAE: 10.31


In [41]:
# Create a table for the tyres + circuit model
tyres_race_errors = pd.DataFrame({
    "session_key": df.loc[X_val.index, "session_key"],
    "race": df.loc[X_val.index, "race_name"],
    "actual": y_val.to_numpy(),
    "predicted": tyres_only_predictions
})

# Calculate the absolute prediction error
tyres_race_errors["absolute_error"] = (
    tyres_race_errors["actual"]
    - tyres_race_errors["predicted"]
).abs()

# Calculate the average error for each validation race
print(
    tyres_race_errors
    .groupby(["session_key", "race"])["absolute_error"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)

session_key  race          
9636         Brazil            21.34
9531         Canada            16.34
9558         United Kingdom    13.48
9955         Spielberg         13.37
10006        Suzuka             9.00
9582         Netherlands        7.89
9221         Lusail             7.89
9496         Japan              6.90
9987         Imola              6.67
9625         Mexico             5.89
9070         Baku               5.27
9157         Monza              2.93


In [42]:
# Add the baseline's absolute errors for the same validation laps
tyres_race_errors["baseline_error"] = (
    tyres_race_errors["actual"].to_numpy()
    - baseline_val_predictions
).__abs__()

# Compare the two models for each race session
race_comparison = (
    tyres_race_errors
    .groupby(["session_key", "race"])
    .agg(
        tyres_rf_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_error", "mean")
    )
)

# Negative values mean the tyres + circuit model is better
race_comparison["difference"] = (
    race_comparison["tyres_rf_mae"]
    - race_comparison["baseline_mae"]
)

# Show the comparison and count how many races the model wins
print(race_comparison.round(2).to_string())

print(
    "\nRaces won by tyres + circuit RF:",
    (race_comparison["difference"] < 0).sum(),
    "out of",
    len(race_comparison)
)

                            tyres_rf_mae  baseline_mae  difference
session_key race                                                  
9070        Baku                    5.27         18.44      -13.18
9157        Monza                   2.93          4.20       -1.27
9221        Lusail                  7.89          6.03        1.86
9496        Japan                   6.90          8.78       -1.88
9531        Canada                 16.34          8.62        7.71
9558        United Kingdom         13.48          4.80        8.68
9582        Netherlands             7.89         14.78       -6.88
9625        Mexico                  5.89         11.47       -5.58
9636        Brazil                 21.34          9.55       11.79
9955        Spielberg              13.37         20.18       -6.82
9987        Imola                   6.67         11.40       -4.73
10006       Suzuka                  9.00          2.24        6.76

Races won by tyres + circuit RF: 7 out of 12


In [43]:
# Add the tyre compound for each validation lap
tyres_race_errors["compound"] = X_val["compound"]

# Compare the model and baseline by tyre compound
tyre_comparison = (
    tyres_race_errors.groupby("compound")
    .agg(
        laps=("absolute_error", "size"),
        rf_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_error", "mean")
    )
)

# Negative values mean our model performs better
tyre_comparison["difference"] = (
    tyre_comparison["rf_mae"]
    - tyre_comparison["baseline_mae"]
)

print(tyre_comparison.round(2).to_string())

              laps  rf_mae  baseline_mae  difference
compound                                            
HARD          5506    6.78         11.03       -4.25
INTERMEDIATE  2069   21.28          8.24       13.04
MEDIUM        4658    8.35         10.31       -1.96
SOFT           503   10.34         10.32        0.02
WET             28   27.51         20.64        6.87


In [44]:
# Select validation laps using dry-weather tyre compounds
dry_laps = tyres_race_errors[
    tyres_race_errors["compound"].isin(["HARD", "MEDIUM", "SOFT"])
]

# Calculate average errors for the model and baseline
dry_rf_mae = dry_laps["absolute_error"].mean()
dry_baseline_mae = dry_laps["baseline_error"].mean()

# Display the comparison
print("Dry-tyre validation laps:", len(dry_laps))
print("Tyres + circuit RF MAE:", round(dry_rf_mae, 2))
print("Baseline MAE:", round(dry_baseline_mae, 2))

Dry-tyre validation laps: 10667
Tyres + circuit RF MAE: 7.63
Baseline MAE: 10.68


In [45]:
# Select laps using intermediate or wet tyres
wet_laps = tyres_race_errors[
    tyres_race_errors["compound"].isin(["INTERMEDIATE", "WET"])
]

# Calculate average prediction errors
wet_rf_mae = wet_laps["absolute_error"].mean()
wet_baseline_mae = wet_laps["baseline_error"].mean()

# Display the results
print("Wet-compound validation laps:", len(wet_laps))
print("Tyres + circuit RF MAE:", round(wet_rf_mae, 2))
print("Baseline MAE:", round(wet_baseline_mae, 2))

Wet-compound validation laps: 2097
Tyres + circuit RF MAE: 21.36
Baseline MAE: 8.41


In [46]:
# Compare actual and predicted lap times for each tyre compound
compound_pace = (
    tyres_race_errors.groupby("compound")[["actual", "predicted"]]
    .mean()
)

# Calculate prediction bias
# Negative values mean the model predicts laps that are too fast
compound_pace["bias"] = (
    compound_pace["predicted"] - compound_pace["actual"]
)

# Display the results
print(compound_pace.round(2).to_string())

              actual  predicted   bias
compound                              
HARD           88.65      91.34   2.68
INTERMEDIATE   91.92     111.35  19.43
MEDIUM         88.30      88.17  -0.12
SOFT           89.79      90.03   0.24
WET           111.89      84.38 -27.51


In [47]:
# Select intermediate-tyre laps from development and validation
dev_intermediate = X_dev["compound"] == "INTERMEDIATE"
val_intermediate = X_val["compound"] == "INTERMEDIATE"

# Compare actual lap durations
print(
    "Development intermediate laps:",
    dev_intermediate.sum()
)
print(
    "Development average lap time:",
    round(y_dev.loc[dev_intermediate].mean(), 2)
)

print(
    "\nValidation intermediate laps:",
    val_intermediate.sum()
)
print(
    "Validation average lap time:",
    round(y_val.loc[val_intermediate].mean(), 2)
)

Development intermediate laps: 1225
Development average lap time: 111.52

Validation intermediate laps: 2069
Validation average lap time: 91.92


In [48]:
# Select intermediate-tyre laps from development
dev_intermediate_pace = (
    df.loc[X_dev.index]
    .loc[X_dev["compound"] == "INTERMEDIATE"]
    .groupby("circuit")["lap_duration"]
    .agg(["count", "mean"])
)

# Select intermediate-tyre laps from validation
val_intermediate_pace = (
    df.loc[X_val.index]
    .loc[X_val["compound"] == "INTERMEDIATE"]
    .groupby("circuit")["lap_duration"]
    .agg(["count", "mean"])
)

# Compare circuits represented in both datasets
print(
    dev_intermediate_pace.join(
        val_intermediate_pace,
        how="inner",
        lsuffix="_dev",
        rsuffix="_val"
    )
    .round(2)
    .to_string()
)

             count_dev  mean_dev  count_val  mean_val
circuit                                              
Silverstone        588    118.08        211    103.02


In [49]:
# Select Silverstone laps using intermediate tyres
dev_silverstone = df.loc[X_dev.index][
    (X_dev["circuit"] == "Silverstone")
    & (X_dev["compound"] == "INTERMEDIATE")
]

val_silverstone = df.loc[X_val.index][
    (X_val["circuit"] == "Silverstone")
    & (X_val["compound"] == "INTERMEDIATE")
]

# Compare average weather conditions between the two groups
weather_comparison = pd.DataFrame({
    "Development": dev_silverstone[
        ["air_temperature", "track_temperature", "rainfall"]
    ].mean(),
    "Validation": val_silverstone[
        ["air_temperature", "track_temperature", "rainfall"]
    ].mean()
})

print(weather_comparison.round(2).to_string())

                   Development  Validation
air_temperature          17.75       14.73
track_temperature        23.09       21.32
rainfall                  0.13        1.00


In [50]:
# Identify the Silverstone intermediate-tyre sessions
silverstone_sessions = pd.concat([
    dev_silverstone.assign(dataset="Development"),
    val_silverstone.assign(dataset="Validation")
])

# Show each session, its season, and its lap count
print(
    silverstone_sessions
    .groupby(["dataset", "season", "session_key"])
    .agg(
        laps=("lap_duration", "size"),
        average_lap=("lap_duration", "mean")
    )
    .round(2)
    .to_string()
)

                                laps  average_lap
dataset     season session_key                   
Development 2025.0 9947          588       118.08
Validation  2024.0 9558          211       103.02


In [51]:
# Compare lap-time distributions for intermediate tyres at Silverstone
silverstone_lap_stats = (
    silverstone_sessions
    .groupby(["dataset", "season"])
    ["lap_duration"]
    .agg(["count", "mean", "median", "min", "max"])
)

# Display the results in seconds
print(silverstone_lap_stats.round(2).to_string())

                    count    mean  median    min     max
dataset     season                                      
Development 2025.0    588  118.08  105.84  97.48  177.05
Validation  2024.0    211  103.02  103.03  95.96  112.54


In [52]:
# Count Silverstone intermediate laps longer than 120 seconds
slow_lap_check = (
    silverstone_sessions
    .assign(slow_lap=lambda data: data["lap_duration"] > 120)
    .groupby(["dataset", "season"])
    .agg(
        total_laps=("lap_duration", "size"),
        slow_laps=("slow_lap", "sum")
    )
)

# Calculate the percentage of unusually slow laps
slow_lap_check["slow_percentage"] = (
    slow_lap_check["slow_laps"]
    / slow_lap_check["total_laps"] * 100
)

print(slow_lap_check.round(2).to_string())

                    total_laps  slow_laps  slow_percentage
dataset     season                                        
Development 2025.0         588        180            30.61
Validation  2024.0         211          0             0.00


In [53]:
# Select unusually slow intermediate laps from Silverstone 2025
slow_silverstone = dev_silverstone[
    dev_silverstone["lap_duration"] > 120
]

# Count how many drivers recorded a slow lap at each lap number
slow_lap_pattern = (
    slow_silverstone
    .groupby("lap_number")
    .agg(
        slow_laps=("lap_duration", "size"),
        average_duration=("lap_duration", "mean")
    )
    .sort_values("slow_laps", ascending=False)
)

# Show the lap numbers with the most slow laps
print(slow_lap_pattern.head(15).round(2).to_string())

            slow_laps  average_duration
lap_number                             
14                 17            136.53
15                 17            147.79
16                 17            164.64
17                 17            172.55
20                 16            162.96
19                 15            155.86
21                 15            157.69
2                  13            137.69
18                 13            127.35
3                  12            136.78
6                  12            136.65
7                  11            122.42
13                  5            121.39


In [54]:
# Group Silverstone 2025 intermediate laps by lap number
silverstone_lap_pace = (
    dev_silverstone
    .groupby("lap_number")["lap_duration"]
    .agg(["count", "median"])
)

# Focus on the suspicious period
print(
    silverstone_lap_pace
    .loc[10:25]
    .round(2)
    .to_string()
)

            count  median
lap_number               
10             10  108.74
11             12  109.48
12              9  108.14
13             16  118.80
14             17  136.57
15             17  144.81
16             17  164.55
17             17  172.53
18             16  125.09
19             15  156.60
20             16  163.07
21             15  157.14
22             16  110.54
23             16  105.88
24             15  104.82
25             15  104.16


In [55]:
# Identify Silverstone 2025 intermediate laps from laps 14 to 21
suspected_laps = dev_silverstone[
    dev_silverstone["lap_number"].between(14, 21)
]

# Count the laps and calculate their typical duration
print("Suspected disruption laps:", len(suspected_laps))
print(
    "Median lap duration:",
    round(suspected_laps["lap_duration"].median(), 2),
    "seconds"
)

# Check how many exceed our exploratory 120-second threshold
print(
    "Laps over 120 seconds:",
    (suspected_laps["lap_duration"] > 120).sum()
)

Suspected disruption laps: 130
Median lap duration: 156.98 seconds
Laps over 120 seconds: 127


In [56]:
# Select all Silverstone 2025 laps during the suspicious period
silverstone_disruption = df[
    (df["session_key"] == 9947)
    & (df["lap_number"].between(14, 21))
]

# Compare lap durations across tyre compounds
print(
    silverstone_disruption
    .groupby("compound")["lap_duration"]
    .agg(["count", "median", "min", "max"])
    .round(2)
    .to_string()
)

              count  median     min     max
compound                                   
INTERMEDIATE    130  156.98  118.04  177.05


In [57]:
# Calculate the original average intermediate lap time
original_average = y_dev[
    X_dev["compound"] == "INTERMEDIATE"
].mean()

# Identify the suspected Silverstone disruption laps
disruption_mask = (
    (df.loc[X_dev.index, "session_key"] == 9947)
    & (df.loc[X_dev.index, "lap_number"].between(14, 21))
)

# Calculate the average without those suspected laps
remaining_intermediate = (
    (X_dev["compound"] == "INTERMEDIATE")
    & ~disruption_mask
)

adjusted_average = y_dev[remaining_intermediate].mean()

# Compare the two averages
print("Original intermediate average:", round(original_average, 2))
print("Without suspected disruption:", round(adjusted_average, 2))
print("Difference:", round(original_average - adjusted_average, 2))

Original intermediate average: 111.52
Without suspected disruption: 106.59
Difference: 4.92


In [58]:
# Identify the suspected Silverstone 2025 disruption laps
disruption_mask = (
    (df.loc[X_dev.index, "session_key"] == 9947)
    & (df.loc[X_dev.index, "lap_number"].between(14, 21))
)

# Keep all other development laps
keep_laps = ~disruption_mask

# Build a fresh model with the same settings
rf_without_disruption = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train without the suspected disruption laps
rf_without_disruption.fit(
    X_dev_tyres.loc[keep_laps],
    y_dev.loc[keep_laps]
)

# Evaluate on the unchanged validation dataset
cleaned_predictions = rf_without_disruption.predict(X_val_tyres)

print("Original tyres + circuit MAE:", round(
    mean_absolute_error(y_val, tyres_only_predictions), 2
))

print("Without suspected disruption MAE:", round(
    mean_absolute_error(y_val, cleaned_predictions), 2
))

# Check whether intermediate-tyre predictions improve
intermediate_mask = X_val["compound"] == "INTERMEDIATE"

print("Original intermediate MAE:", round(
    mean_absolute_error(
        y_val.loc[intermediate_mask],
        tyres_only_predictions[intermediate_mask.to_numpy()]
    ), 2
))

print("New intermediate MAE:", round(
    mean_absolute_error(
        y_val.loc[intermediate_mask],
        cleaned_predictions[intermediate_mask.to_numpy()]
    ), 2
))

Original tyres + circuit MAE: 9.89
Without suspected disruption MAE: 8.33
Original intermediate MAE: 21.28
New intermediate MAE: 16.05


In [59]:
# Compare the original and revised model predictions
compound_results = pd.DataFrame({
    "compound": X_val["compound"].to_numpy(),
    "actual": y_val.to_numpy(),
    "original_prediction": tyres_only_predictions,
    "revised_prediction": cleaned_predictions
})

# Calculate absolute errors for both models
compound_results["original_error"] = (
    compound_results["actual"]
    - compound_results["original_prediction"]
).abs()

compound_results["revised_error"] = (
    compound_results["actual"]
    - compound_results["revised_prediction"]
).abs()

# Calculate average error for each tyre compound
print(
    compound_results
    .groupby("compound")[["original_error", "revised_error"]]
    .mean()
    .round(2)
    .to_string()
)

              original_error  revised_error
compound                                   
HARD                    6.78           5.98
INTERMEDIATE           21.28          16.05
MEDIUM                  8.35           7.45
SOFT                   10.34           9.41
WET                    27.51          26.34


In [60]:
# Select the suspected Silverstone 2025 disruption period
disruption_laps = df[
    (df["session_key"] == 9947)
    & (df["lap_number"].between(14, 21))
]

# Count affected drivers and laps
print("Distinct drivers:", disruption_laps["driver_number"].nunique())
print("Total laps:", len(disruption_laps))

# Count how many drivers recorded laps over 120 seconds
slow_drivers = disruption_laps[
    disruption_laps["lap_duration"] > 120
]["driver_number"].nunique()

print("Drivers with slow laps:", slow_drivers)

Distinct drivers: 17
Total laps: 130
Drivers with slow laps: 17


In [61]:
# Look for columns related to race control or safety cars
race_control_columns = [
    column for column in df.columns
    if any(
        word in column.lower()
        for word in ["safety", "control", "flag", "status"]
    )
]

# Display any matching columns
print("Race-control-related columns:", race_control_columns)

Race-control-related columns: []


In [62]:
# Import requests to retrieve data from OpenF1
import requests

# Retrieve race-control messages for Silverstone 2025
url = "https://api.openf1.org/v1/race_control"

response = requests.get(
    url,
    params={"session_key": 9947},
    timeout=30
)

# Check that the request succeeded
response.raise_for_status()

# Convert the response into a pandas DataFrame
silverstone_race_control = pd.DataFrame(response.json())

# Display messages mentioning safety cars or red flags
print(
    silverstone_race_control[
        silverstone_race_control["message"].str.contains(
            "SAFETY CAR|VIRTUAL SAFETY CAR|RED FLAG",
            case=False,
            na=False
        )
    ][["date", "message"]].to_string(index=False)
)

                     date                                                                                          message
2025-07-06T13:55:00+00:00                                              FORMATION LAP WILL BE STARTED BEHIND THE SAFETY CAR
2025-07-06T14:05:48+00:00                                                                      VIRTUAL SAFETY CAR DEPLOYED
2025-07-06T14:10:18+00:00                                                                        VIRTUAL SAFETY CAR ENDING
2025-07-06T14:13:39+00:00                                                                      VIRTUAL SAFETY CAR DEPLOYED
2025-07-06T14:17:04+00:00                                                                        VIRTUAL SAFETY CAR ENDING
2025-07-06T14:29:14+00:00                                                                              SAFETY CAR DEPLOYED
2025-07-06T14:38:36+00:00                                                                           SAFETY CAR IN THIS LAP
2025-07-06T14:41

In [63]:
# Select the suspected Silverstone disruption laps
suspected_silverstone_laps = df[
    (df["session_key"] == 9947)
    & (df["lap_number"].between(14, 21))
].copy()

# Convert lap timestamps to UTC
suspected_silverstone_laps["lap_time_utc"] = pd.to_datetime(
    suspected_silverstone_laps["date_start"],
    utc=True,
    format="mixed"
)

# Show the timestamp range for each lap number
print(
    suspected_silverstone_laps
    .groupby("lap_number")["lap_time_utc"]
    .agg(["min", "max"])
    .to_string()
)

                                        min                              max
lap_number                                                                  
14         2025-07-06 14:28:55.335000+00:00 2025-07-06 14:29:47.286000+00:00
15         2025-07-06 14:31:11.167000+00:00 2025-07-06 14:32:04.128000+00:00
16         2025-07-06 14:34:01.281000+00:00 2025-07-06 14:34:21.296000+00:00
17         2025-07-06 14:36:46.761000+00:00 2025-07-06 14:37:05.848000+00:00
18         2025-07-06 14:39:43.652000+00:00 2025-07-06 14:39:54.304000+00:00
19         2025-07-06 14:41:41.662000+00:00 2025-07-06 14:42:06.603000+00:00
20         2025-07-06 14:44:21.273000+00:00 2025-07-06 14:44:38.685000+00:00
21         2025-07-06 14:47:04.296000+00:00 2025-07-06 14:47:23.202000+00:00


In [64]:
# Copy the suspected Silverstone laps
lap_overlap = suspected_silverstone_laps.copy()

# Calculate when each lap ended
lap_overlap["lap_end_utc"] = (
    lap_overlap["lap_time_utc"]
    + pd.to_timedelta(lap_overlap["lap_duration"], unit="s")
)

# Define the two confirmed Safety Car periods
sc_periods = [
    ("2025-07-06 14:29:14+00:00", "2025-07-06 14:38:36+00:00"),
    ("2025-07-06 14:41:15+00:00", "2025-07-06 14:49:02+00:00")
]

# Start with no laps marked as overlapping
lap_overlap["safety_car_overlap"] = False

# Check whether each lap overlaps either Safety Car period
for start, end in sc_periods:
    sc_start = pd.Timestamp(start)
    sc_end = pd.Timestamp(end)

    lap_overlap["safety_car_overlap"] |= (
        (lap_overlap["lap_time_utc"] < sc_end)
        & (lap_overlap["lap_end_utc"] > sc_start)
    )

# Count the overlapping laps by lap number
print(
    lap_overlap.groupby("lap_number")["safety_car_overlap"]
    .agg(["sum", "count"])
    .to_string()
)

print(
    "\nTotal overlapping laps:",
    lap_overlap["safety_car_overlap"].sum()
)

            sum  count
lap_number            
14           17     17
15           17     17
16           17     17
17           17     17
18           16     16
19           15     15
20           16     16
21           15     15

Total overlapping laps: 130


In [65]:
# Get the indices of laps overlapping the confirmed Safety Car periods
confirmed_sc_indices = lap_overlap.loc[
    lap_overlap["safety_car_overlap"]
].index

# Count how many appear in each modelling dataset
print(
    "Development laps overlapping Safety Car:",
    X_dev.index.isin(confirmed_sc_indices).sum()
)

print(
    "Validation laps overlapping Safety Car:",
    X_val.index.isin(confirmed_sc_indices).sum()
)

print(
    "Original development laps:",
    len(X_dev)
)

Development laps overlapping Safety Car: 130
Validation laps overlapping Safety Car: 0
Original development laps: 46050


In [66]:
# Get the unique race sessions in our development dataset
development_sessions = (
    df.loc[X_dev.index, "session_key"]
    .drop_duplicates()
    .sort_values()
)

# Count the sessions we need to investigate
print("Development race sessions:", len(development_sessions))

# Confirm that Silverstone 2025 is included
print(
    "Silverstone 2025 included:",
    9947 in development_sessions.values
)

Development race sessions: 44
Silverstone 2025 included: True


In [68]:
# Choose a development race other than Silverstone 2025
test_session = development_sessions[
    development_sessions != 9947
].iloc[0]

# Request race-control messages for that race
response = requests.get(
    "https://api.openf1.org/v1/race_control",
    params={"session_key": int(test_session)},
    timeout=30
)

# Display the result
print("Session key:", test_session)
print("HTTP status:", response.status_code)

if response.ok:
    print("Race-control messages:", len(response.json()))
else:
    print(response.text[:200])

Session key: 7787
HTTP status: 200
Race-control messages: 111


In [69]:
# Store race-control messages and unsuccessful sessions
race_control_records = []
failed_sessions = []

# Retrieve race-control messages for each development race
for session in development_sessions:
    response = requests.get(
        "https://api.openf1.org/v1/race_control",
        params={"session_key": int(session)},
        timeout=30
    )

    # Save successful responses
    if response.ok:
        race_control_records.extend(response.json())
    else:
        # Record sessions that need investigation
        failed_sessions.append((int(session), response.status_code))

# Combine all retrieved messages into one DataFrame
development_race_control = pd.DataFrame(race_control_records)

# Check the retrieval results
print("Development races requested:", len(development_sessions))
print("Races with records:",
      development_race_control["session_key"].nunique())
print("Total race-control messages:", len(development_race_control))
print("Unsuccessful requests:", failed_sessions)

Development races requested: 44
Races with records: 44
Total race-control messages: 4057
Unsuccessful requests: []


In [70]:
# Find messages that mention Safety Cars or Virtual Safety Cars
safety_messages = development_race_control[
    development_race_control["message"].str.contains(
        "SAFETY CAR",
        case=False,
        na=False
    )
].copy()

# Count the different messages
print(
    safety_messages["message"]
    .value_counts()
    .head(25)
    .to_string()
)

message
SAFETY CAR DEPLOYED                                                                                                                                                            30
SAFETY CAR IN THIS LAP                                                                                                                                                         25
VIRTUAL SAFETY CAR DEPLOYED                                                                                                                                                    24
VIRTUAL SAFETY CAR ENDING                                                                                                                                                      21
ALL CARS MAY OVERTAKE THE SAFETY CAR                                                                                                                                            4
SAFETY CAR THROUGH THE PIT LANE                                                                       

In [71]:
# Define the four exact messages we want to investigate
event_messages = [
    "SAFETY CAR DEPLOYED",
    "SAFETY CAR IN THIS LAP",
    "VIRTUAL SAFETY CAR DEPLOYED",
    "VIRTUAL SAFETY CAR ENDING"
]

# Keep only actual deployment and ending messages
sc_events = development_race_control[
    development_race_control["message"].isin(event_messages)
].copy()

# Count each event type separately for every race
event_counts = pd.crosstab(
    sc_events["session_key"],
    sc_events["message"]
).reindex(columns=event_messages, fill_value=0)

# Find races where deployment and ending counts differ
mismatched_races = event_counts[
    (event_counts["SAFETY CAR DEPLOYED"] !=
     event_counts["SAFETY CAR IN THIS LAP"])
    |
    (event_counts["VIRTUAL SAFETY CAR DEPLOYED"] !=
     event_counts["VIRTUAL SAFETY CAR ENDING"])
]

# Display the results
print("Races needing investigation:", len(mismatched_races))
print(mismatched_races.to_string())

Races needing investigation: 7
message      SAFETY CAR DEPLOYED  SAFETY CAR IN THIS LAP  VIRTUAL SAFETY CAR DEPLOYED  VIRTUAL SAFETY CAR ENDING
session_key                                                                                                     
7787                           3                       1                            1                          1
9181                           1                       0                            1                          1
9205                           1                       0                            0                          0
9488                           0                       0                            2                          1
9598                           0                       0                            1                          0
9655                           3                       3                            1                          0
9963                           1                       0         

In [72]:
# Select race-control messages for session 7787
race_7787 = development_race_control[
    development_race_control["session_key"] == 7787
].copy()

# Find messages related to interruptions and resumptions
important_messages = race_7787[
    race_7787["message"].str.contains(
        "SAFETY CAR|RED FLAG|SUSPEND|RESUME|RESTART",
        case=False,
        na=False
    )
]

# Display the messages in chronological order
print(
    important_messages
    .sort_values("date")[["date", "message"]]
    .to_string(index=False)
)

                     date                                                              message
2023-04-02T05:04:27+00:00                                                  SAFETY CAR DEPLOYED
2023-04-02T05:07:27+00:00                                               SAFETY CAR IN THIS LAP
2023-04-02T05:14:24+00:00                                                  SAFETY CAR DEPLOYED
2023-04-02T05:17:02+00:00                                                             RED FLAG
2023-04-02T05:22:27+00:00                                            RACE WILL RESUME AT 15:33
2023-04-02T05:32:01+00:00                 RACE WILL RESUME AT 15:33 - STANDING START PROCEDURE
2023-04-02T05:36:23+00:00                                RESTART PROCEDURE UNDER INVESTIGATION
2023-04-02T05:46:30+00:00                                 RESTART PROCEDURE NO FURTHER ACTION 
2023-04-02T05:48:36+00:00                                          VIRTUAL SAFETY CAR DEPLOYED
2023-04-02T05:50:38+00:00                         

In [73]:
# Get the remaining races with unmatched Safety Car events
remaining_sessions = mismatched_races.index[
    mismatched_races.index != 7787
]

# Count red flags in each remaining race
for session in remaining_sessions:
    race_messages = development_race_control[
        development_race_control["session_key"] == session
    ]

    red_flags = (
        race_messages["message"].str.strip().str.upper() == "RED FLAG"
    ).sum()

    print(f"Session {session}: {red_flags} red flag(s)")

Session 9181: 1 red flag(s)
Session 9205: 1 red flag(s)
Session 9488: 0 red flag(s)
Session 9598: 0 red flag(s)
Session 9655: 0 red flag(s)
Session 9963: 0 red flag(s)


In [74]:
# Select race-control messages for session 9488
race_9488 = development_race_control[
    development_race_control["session_key"] == 9488
].copy()

# Find messages about VSC, Safety Car and red flags
important_messages = race_9488[
    race_9488["message"].str.contains(
        "VIRTUAL SAFETY CAR|SAFETY CAR DEPLOYED|RED FLAG",
        case=False,
        na=False
    )
]

# Display the messages in chronological order
print(
    important_messages
    .sort_values("date")[["date", "message"]]
    .to_string(index=False)
)

                     date                     message
2024-03-24T04:26:24+00:00 VIRTUAL SAFETY CAR DEPLOYED
2024-03-24T04:27:29+00:00   VIRTUAL SAFETY CAR ENDING
2024-03-24T05:22:02+00:00 VIRTUAL SAFETY CAR DEPLOYED
2024-03-24T05:23:37+00:00              CHEQUERED FLAG


In [75]:
# Select race-control messages for session 9598
race_9598 = development_race_control[
    development_race_control["session_key"] == 9598
].copy()

# Keep messages related to interruptions or the race finish
important_messages = race_9598[
    race_9598["message"].str.contains(
        "VIRTUAL SAFETY CAR|SAFETY CAR|RED FLAG|CHEQUERED FLAG",
        case=False,
        na=False
    )
]

# Display the timeline in chronological order
print(
    important_messages
    .sort_values("date")[["date", "message"]]
    .to_string(index=False)
)

                     date                                                                                                             message
2024-09-15T12:34:38+00:00                                                                                         VIRTUAL SAFETY CAR DEPLOYED
2024-09-15T12:37:07+00:00                                                                                                      CHEQUERED FLAG
2024-09-15T12:46:47+00:00                                             INCIDENT INVOLVING CAR 27 (HUL) NOTED - VIRTUAL SAFETY CAR INFRINGEMENT
2024-09-15T12:46:58+00:00 FIA STEWARDS: INCIDENT INVOLVING CAR 27 (HUL) WILL BE INVESTIGATED AFTER THE RACE - VIRTUAL SAFETY CAR INFRINGEMENT
2024-09-15T12:47:38+00:00                                              INCIDENT INVOLVING CAR 1 (VER) NOTED - VIRTUAL SAFETY CAR INFRINGEMENT
2024-09-15T12:47:49+00:00  FIA STEWARDS: INCIDENT INVOLVING CAR 1 (VER) WILL BE INVESTIGATED AFTER THE RACE - VIRTUAL SAFETY CAR INFRINGEMENT
2024-0

In [76]:
# Select race-control messages for session 9655
race_9655 = development_race_control[
    development_race_control["session_key"] == 9655
].copy()

# Keep actual race-control events and race-finish messages
event_messages = [
    "SAFETY CAR DEPLOYED",
    "SAFETY CAR IN THIS LAP",
    "VIRTUAL SAFETY CAR DEPLOYED",
    "VIRTUAL SAFETY CAR ENDING",
    "RED FLAG",
    "CHEQUERED FLAG"
]

# Display the event timeline in chronological order
print(
    race_9655[
        race_9655["message"].str.strip().isin(event_messages)
    ]
    .sort_values("date")[["date", "message"]]
    .to_string(index=False)
)

                     date                     message
2024-12-01T16:04:33+00:00         SAFETY CAR DEPLOYED
2024-12-01T16:10:56+00:00      SAFETY CAR IN THIS LAP
2024-12-01T16:56:32+00:00         SAFETY CAR DEPLOYED
2024-12-01T17:05:22+00:00      SAFETY CAR IN THIS LAP
2024-12-01T17:08:24+00:00 VIRTUAL SAFETY CAR DEPLOYED
2024-12-01T17:08:44+00:00         SAFETY CAR DEPLOYED
2024-12-01T17:12:10+00:00      SAFETY CAR IN THIS LAP
2024-12-01T17:34:38+00:00              CHEQUERED FLAG


In [77]:
# Select race-control messages for session 9963
race_9963 = development_race_control[
    development_race_control["session_key"] == 9963
].copy()

# Keep messages about interruptions and the race finish
important_messages = race_9963[
    race_9963["message"].str.contains(
        "SAFETY CAR|RED FLAG|CHEQUERED FLAG",
        case=False,
        na=False
    )
]

# Display the messages in chronological order
print(
    important_messages
    .sort_values("date")[["date", "message"]]
    .to_string(index=False)
)

                     date                                                                                                                                                                     message
2025-06-15T19:27:32+00:00                                                                                                                                                         SAFETY CAR DEPLOYED
2025-06-15T19:27:49+00:00                                                                                                                      ALL CARS TO FOLLOW THE SAFETY CAR THROUGH THE PIT LANE
2025-06-15T19:34:56+00:00                                                                                                                                                              CHEQUERED FLAG
2025-06-15T19:48:27+00:00                                             INCIDENT INVOLVING CARS 12 (ANT), 81 (PIA), 31 (OCO), 16 (LEC), 55 (SAI), 10 (GAS) AND 18 (STR) NOTED - SAFETY CAR INFRINGEMENT
2025-06-15

In [78]:
# Inspect the final two races with unmatched Safety Car events
for session in [9181, 9205]:
    # Select this race's messages
    race_messages = development_race_control[
        development_race_control["session_key"] == session
    ]

    # Keep only the key interruption events
    key_events = race_messages[
        race_messages["message"].str.strip().isin([
            "SAFETY CAR DEPLOYED",
            "SAFETY CAR IN THIS LAP",
            "VIRTUAL SAFETY CAR DEPLOYED",
            "VIRTUAL SAFETY CAR ENDING",
            "RED FLAG",
            "CHEQUERED FLAG"
        ])
    ]

    # Display the events in time order
    print(f"\nSession {session}")
    print(
        key_events
        .sort_values("date")[["date", "message"]]
        .to_string(index=False)
    )


Session 9181
                     date                     message
2023-10-29T20:09:05+00:00 VIRTUAL SAFETY CAR DEPLOYED
2023-10-29T20:10:21+00:00   VIRTUAL SAFETY CAR ENDING
2023-10-29T20:49:24+00:00         SAFETY CAR DEPLOYED
2023-10-29T20:51:53+00:00                    RED FLAG
2023-10-29T22:05:49+00:00              CHEQUERED FLAG

Session 9205
                     date             message
2023-11-05T17:03:50+00:00 SAFETY CAR DEPLOYED
2023-11-05T17:06:35+00:00            RED FLAG
2023-11-05T19:00:12+00:00      CHEQUERED FLAG


In [79]:
# Make a copy so we preserve the original records
race_control_timeline = development_race_control.copy()

# Convert race-control timestamps to UTC
race_control_timeline["event_time"] = pd.to_datetime(
    race_control_timeline["date"],
    utc=True,
    format="mixed"
)

# Check that every timestamp converted successfully
print("Total messages:", len(race_control_timeline))
print("Missing timestamps:", race_control_timeline["event_time"].isna().sum())
print("Earliest message:", race_control_timeline["event_time"].min())
print("Latest message:", race_control_timeline["event_time"].max())

Total messages: 4057
Missing timestamps: 0
Earliest message: 2023-03-05 14:04:34+00:00
Latest message: 2025-12-07 14:38:08+00:00


In [80]:
# Define messages that change race-control status
status_messages = [
    "SAFETY CAR DEPLOYED",
    "SAFETY CAR IN THIS LAP",
    "VIRTUAL SAFETY CAR DEPLOYED",
    "VIRTUAL SAFETY CAR ENDING",
    "RED FLAG",
    "CHEQUERED FLAG"
]

# Keep only these messages
status_events = race_control_timeline[
    race_control_timeline["message"].str.strip().isin(status_messages)
].copy()

# Sort events by race and timestamp
status_events = status_events.sort_values(
    ["session_key", "event_time"]
)

# Check how many events we have
print("Status-change events:", len(status_events))
print("Races represented:", status_events["session_key"].nunique())

# Check the different event types
print(status_events["message"].str.strip().value_counts())

Status-change events: 152
Races represented: 44
message
CHEQUERED FLAG                 44
SAFETY CAR DEPLOYED            30
SAFETY CAR IN THIS LAP         25
VIRTUAL SAFETY CAR DEPLOYED    24
VIRTUAL SAFETY CAR ENDING      21
RED FLAG                        8
Name: count, dtype: int64


In [81]:
# Store the interruption intervals
disruption_intervals = []

# Process each race separately
for session, events in status_events.groupby("session_key"):
    active_type = None
    start_time = None

    for _, event in events.iterrows():
        message = event["message"].strip()
        event_time = event["event_time"]

        # Start a Safety Car or VSC period
        if message in [
            "SAFETY CAR DEPLOYED",
            "VIRTUAL SAFETY CAR DEPLOYED"
        ]:
            # Close an existing period if another deployment replaces it
            if active_type is not None:
                disruption_intervals.append({
                    "session_key": session,
                    "type": active_type,
                    "start": start_time,
                    "end": event_time
                })

            active_type = (
                "SC" if message == "SAFETY CAR DEPLOYED" else "VSC"
            )
            start_time = event_time

        # End an active period at its matching ending or a race stoppage
        elif active_type is not None:
            matching_end = (
                (active_type == "SC" and message == "SAFETY CAR IN THIS LAP")
                or
                (active_type == "VSC" and message == "VIRTUAL SAFETY CAR ENDING")
            )

            if matching_end or message in ["RED FLAG", "CHEQUERED FLAG"]:
                disruption_intervals.append({
                    "session_key": session,
                    "type": active_type,
                    "start": start_time,
                    "end": event_time
                })

                active_type = None
                start_time = None

# Convert the intervals into a DataFrame
disruption_intervals = pd.DataFrame(disruption_intervals)

# Inspect the results
print("Total intervals:", len(disruption_intervals))
print(disruption_intervals["type"].value_counts())
print("Races with intervals:", disruption_intervals["session_key"].nunique())

Total intervals: 54
type
SC     30
VSC    24
Name: count, dtype: int64
Races with intervals: 28


In [82]:
# Find intervals that end at or before they start
invalid_intervals = disruption_intervals[
    disruption_intervals["end"] <= disruption_intervals["start"]
]

# Calculate the duration of each interval in minutes
disruption_intervals["duration_minutes"] = (
    disruption_intervals["end"] - disruption_intervals["start"]
).dt.total_seconds() / 60

# Check the results
print("Invalid intervals:", len(invalid_intervals))
print("Shortest interval (minutes):",
      round(disruption_intervals["duration_minutes"].min(), 2))
print("Longest interval (minutes):",
      round(disruption_intervals["duration_minutes"].max(), 2))

Invalid intervals: 0
Shortest interval (minutes): 0.33
Longest interval (minutes): 15.68


In [83]:
# Copy the original records for our development laps
development_laps = df.loc[X_dev.index].copy()

# Convert each lap's start timestamp to UTC
development_laps["lap_start_utc"] = pd.to_datetime(
    development_laps["date_start"],
    utc=True,
    format="mixed"
)

# Calculate when each lap finished
development_laps["lap_end_utc"] = (
    development_laps["lap_start_utc"]
    + pd.to_timedelta(development_laps["lap_duration"], unit="s")
)

# Check that the timestamps are usable
print("Development laps:", len(development_laps))
print("Missing start times:", development_laps["lap_start_utc"].isna().sum())
print("Missing end times:", development_laps["lap_end_utc"].isna().sum())

Development laps: 46050
Missing start times: 0
Missing end times: 0


In [84]:
# Start with every development lap marked as unaffected
development_laps["sc_vsc_overlap"] = False

# Check each confirmed Safety Car or VSC interval
for _, interval in disruption_intervals.iterrows():

    # A lap must belong to the same race and overlap the interval
    overlaps = (
        (development_laps["session_key"] == interval["session_key"])
        & (development_laps["lap_start_utc"] < interval["end"])
        & (development_laps["lap_end_utc"] > interval["start"])
    )

    # Flag overlapping laps without removing them
    development_laps.loc[overlaps, "sc_vsc_overlap"] = True

# Summarise the results
print("Development laps:", len(development_laps))
print("SC/VSC-overlapping laps:",
      development_laps["sc_vsc_overlap"].sum())
print("Races with overlapping laps:",
      development_laps.loc[
          development_laps["sc_vsc_overlap"], "session_key"
      ].nunique())

# Confirm our original Silverstone finding
silverstone_overlap = development_laps[
    (development_laps["session_key"] == 9947)
    & (development_laps["lap_number"].between(14, 21))
]

print("Silverstone laps 14–21 flagged:",
      silverstone_overlap["sc_vsc_overlap"].sum())

Development laps: 46050
SC/VSC-overlapping laps: 2471
Races with overlapping laps: 28
Silverstone laps 14–21 flagged: 130


In [85]:
# Compare lap durations for flagged and unflagged laps
lap_comparison = development_laps.groupby("sc_vsc_overlap")[
    "lap_duration"
].agg(["count", "mean", "median"])

# Make the output easier to read
lap_comparison = lap_comparison.round(2)

print(lap_comparison)

                count    mean  median
sc_vsc_overlap                       
False           43579   89.46   86.52
True             2471  122.92  121.16


In [86]:
# Count flagged and unflagged laps for each tyre compound
compound_overlap = pd.crosstab(
    development_laps["compound"],
    development_laps["sc_vsc_overlap"]
)

# Label the columns clearly
compound_overlap.columns = ["No SC/VSC overlap", "SC/VSC overlap"]

# Calculate the percentage flagged within each compound
compound_overlap["Flagged %"] = (
    100
    * compound_overlap["SC/VSC overlap"]
    / compound_overlap.sum(axis=1)
).round(2)

# Display the comparison
print(compound_overlap.to_string())

              No SC/VSC overlap  SC/VSC overlap  Flagged %
compound                                                  
HARD                      20761             919       4.24
INTERMEDIATE                877             348      28.41
MEDIUM                    16563             897       5.14
SOFT                       5378             307       5.40


In [87]:
# Select intermediate-tyre laps only
intermediate_laps = development_laps[
    development_laps["compound"] == "INTERMEDIATE"
]

# Compare lap times with and without SC/VSC overlap
print(
    intermediate_laps.groupby("sc_vsc_overlap")["lap_duration"]
    .agg(["count", "mean", "median"])
    .round(2)
)

                count    mean  median
sc_vsc_overlap                       
False             877  100.28   96.27
True              348  139.85  137.97


In [88]:
# Select red-flag messages from our development races
red_flag_events = race_control_timeline[
    race_control_timeline["message"].str.strip() == "RED FLAG"
].copy()

# Count the red flags and affected races
print("Red flag messages:", len(red_flag_events))
print("Races with red flags:", red_flag_events["session_key"].nunique())

# Display the affected race sessions
print("Session keys:", sorted(red_flag_events["session_key"].unique()))

Red flag messages: 8
Races with red flags: 5
Session keys: [7787, 9181, 9205, 9523, 9939]


In [89]:
# Examine red flags and race-resumption messages
red_flag_sessions = [7787, 9181, 9205, 9523, 9939]

# Select messages from the five affected races
red_flag_timeline = race_control_timeline[
    race_control_timeline["session_key"].isin(red_flag_sessions)
    & race_control_timeline["message"].str.contains(
        "RED FLAG|RESUME|RESTART|STANDING START|ROLLING START",
        case=False,
        na=False
    )
].copy()

# Display the messages in chronological order
print(
    red_flag_timeline[
        ["session_key", "event_time", "message"]
    ]
    .sort_values(["session_key", "event_time"])
    .to_string(index=False)
)

 session_key                event_time                                                                               message
        7787 2023-04-02 05:17:02+00:00                                                                              RED FLAG
        7787 2023-04-02 05:22:27+00:00                                                             RACE WILL RESUME AT 15:33
        7787 2023-04-02 05:32:01+00:00                                  RACE WILL RESUME AT 15:33 - STANDING START PROCEDURE
        7787 2023-04-02 05:34:05+00:00                                                              STANDING START PROCEDURE
        7787 2023-04-02 05:36:23+00:00                                                 RESTART PROCEDURE UNDER INVESTIGATION
        7787 2023-04-02 05:46:30+00:00                                                  RESTART PROCEDURE NO FURTHER ACTION 
        7787 2023-04-02 06:41:07+00:00                                                                              RED FLAG


In [90]:
# Select messages that indicate a race restart
restart_messages = [
    "STANDING START",
    "STANDING START PROCEDURE",
    "ROLLING START"
]

# Find the restart events in our five red-flagged races
restart_events = race_control_timeline[
    race_control_timeline["session_key"].isin(
        red_flag_events["session_key"].unique()
    )
    & race_control_timeline["message"].str.strip().isin(
        restart_messages
    )
].copy()

# Display the restart timestamps
print(
    restart_events[
        ["session_key", "event_time", "message"]
    ]
    .sort_values(["session_key", "event_time"])
    .to_string(index=False)
)

 session_key                event_time                  message
        7787 2023-04-02 05:34:05+00:00 STANDING START PROCEDURE
        7787 2023-04-02 06:57:13+00:00 STANDING START PROCEDURE
        9181 2023-10-29 21:13:38+00:00 STANDING START PROCEDURE
        9523 2024-05-26 13:44:53+00:00           STANDING START
        9939 2025-07-27 14:29:35+00:00            ROLLING START


In [91]:
# Check each red flag against the next recorded restart
for _, flag in red_flag_events.sort_values("event_time").iterrows():

    # Find restart markers later in the same race
    later_restarts = restart_events[
        (restart_events["session_key"] == flag["session_key"])
        & (restart_events["event_time"] > flag["event_time"])
    ]

    # Find the first subsequent restart, if one exists
    next_restart = (
        later_restarts["event_time"].min()
        if not later_restarts.empty
        else "No restart marker"
    )

    print(
        f"Session {flag['session_key']} | "
        f"Red flag: {flag['event_time']} | "
        f"Next restart: {next_restart}"
    )

Session 7787 | Red flag: 2023-04-02 05:17:02+00:00 | Next restart: 2023-04-02 05:34:05+00:00
Session 7787 | Red flag: 2023-04-02 06:41:07+00:00 | Next restart: 2023-04-02 06:57:13+00:00
Session 7787 | Red flag: 2023-04-02 07:00:26+00:00 | Next restart: No restart marker
Session 7787 | Red flag: 2023-04-02 07:37:06+00:00 | Next restart: No restart marker
Session 9181 | Red flag: 2023-10-29 20:51:53+00:00 | Next restart: 2023-10-29 21:13:38+00:00
Session 9205 | Red flag: 2023-11-05 17:06:35+00:00 | Next restart: No restart marker
Session 9523 | Red flag: 2024-05-26 13:04:08+00:00 | Next restart: 2024-05-26 13:44:53+00:00
Session 9939 | Red flag: 2025-07-27 13:01:58+00:00 | Next restart: 2025-07-27 14:29:35+00:00


In [92]:
# Select messages for session 9205
race_9205 = race_control_timeline[
    race_control_timeline["session_key"] == 9205
].copy()

# Focus on the period surrounding the scheduled restart
restart_window = race_9205[
    race_9205["event_time"].between(
        pd.Timestamp("2023-11-05 17:25:00+00:00"),
        pd.Timestamp("2023-11-05 17:40:00+00:00")
    )
]

# Display all messages, not just those mentioning restarts
print(
    restart_window[["event_time", "message"]]
    .sort_values("event_time")
    .to_string(index=False)
)

                      event_time                                                                                                                          message
       2023-11-05 17:27:51+00:00                                                                                                                      TRACK CLEAR
       2023-11-05 17:29:01+00:00                                                                             RACE WILL RESUME AT 14:31 - STANDING START PROCEDURE
       2023-11-05 17:30:46+00:00                                                                                             ALL CARS MAY OVERTAKE THE SAFETY CAR
2023-11-05 17:31:00.366000+00:00                                                                                                                  SESSION STARTED
       2023-11-05 17:35:37+00:00 FIA STEWARDS: LAP 1 TURN 1 INCIDENT INVOLVING CARS 23 (ALB) AND 20 (MAG) REVIEWED NO FURTHER INVESTIGATION - CAUSING A COLLISION
       2023-11-05 17:36:37+0

In [93]:
# Find session-start messages in our five red-flagged races
session_starts = race_control_timeline[
    race_control_timeline["session_key"].isin(
        red_flag_events["session_key"].unique()
    )
    & (
        race_control_timeline["message"].str.strip()
        == "SESSION STARTED"
    )
].copy()

# Display the recorded session-start timestamps
print(
    session_starts[
        ["session_key", "event_time", "message"]
    ]
    .sort_values(["session_key", "event_time"])
    .to_string(index=False)
)

 session_key                       event_time         message
        7787 2023-04-02 05:03:18.684000+00:00 SESSION STARTED
        7787 2023-04-02 05:33:00.429000+00:00 SESSION STARTED
        7787 2023-04-02 06:56:00.419000+00:00 SESSION STARTED
        7787 2023-04-02 07:33:00.456000+00:00 SESSION STARTED
        9181 2023-10-29 20:03:18.321000+00:00 SESSION STARTED
        9181 2023-10-29 21:13:00.310000+00:00 SESSION STARTED
        9205 2023-11-05 17:03:22.841000+00:00 SESSION STARTED
        9205 2023-11-05 17:31:00.366000+00:00 SESSION STARTED
        9523 2024-05-26 13:03:11.068000+00:00 SESSION STARTED
        9523 2024-05-26 13:44:00.121000+00:00 SESSION STARTED
        9939 2025-07-27 14:20:00.065000+00:00 SESSION STARTED


In [94]:
# Store the red-flag stoppage intervals
red_flag_intervals = []

# Process each race separately
for session, flags in red_flag_events.groupby("session_key"):

    # Find the chequered flag for this race
    finish_time = status_events.loc[
        (status_events["session_key"] == session)
        & (status_events["message"].str.strip() == "CHEQUERED FLAG"),
        "event_time"
    ].min()

    # Find session-start messages for this race
    starts = session_starts.loc[
        session_starts["session_key"] == session,
        "event_time"
    ]

    for _, flag in flags.iterrows():
        flag_time = flag["event_time"]

        # Ignore red flags recorded after the race finished
        if flag_time >= finish_time:
            continue

        # Find the next session start after this red flag
        later_starts = starts[starts > flag_time]

        if not later_starts.empty:
            red_flag_intervals.append({
                "session_key": session,
                "start": flag_time,
                "end": later_starts.min()
            })

# Convert the intervals to a DataFrame
red_flag_intervals = pd.DataFrame(red_flag_intervals)

# Check the results
print("Red-flag intervals:", len(red_flag_intervals))
print("Races represented:", red_flag_intervals["session_key"].nunique())
print(red_flag_intervals.to_string(index=False))


Red-flag intervals: 7
Races represented: 5
 session_key                     start                              end
        7787 2023-04-02 05:17:02+00:00 2023-04-02 05:33:00.429000+00:00
        7787 2023-04-02 06:41:07+00:00 2023-04-02 06:56:00.419000+00:00
        7787 2023-04-02 07:00:26+00:00 2023-04-02 07:33:00.456000+00:00
        9181 2023-10-29 20:51:53+00:00 2023-10-29 21:13:00.310000+00:00
        9205 2023-11-05 17:06:35+00:00 2023-11-05 17:31:00.366000+00:00
        9523 2024-05-26 13:04:08+00:00 2024-05-26 13:44:00.121000+00:00
        9939 2025-07-27 13:01:58+00:00 2025-07-27 14:20:00.065000+00:00


In [95]:
# Start with no development laps flagged for red-flag overlap
development_laps["red_flag_overlap"] = False

# Check each recorded red-flag stoppage
for _, interval in red_flag_intervals.iterrows():

    # Match laps from the same race whose time intervals overlap
    overlaps = (
        (development_laps["session_key"] == interval["session_key"])
        & (development_laps["lap_start_utc"] < interval["end"])
        & (development_laps["lap_end_utc"] > interval["start"])
    )

    # Flag the overlapping laps
    development_laps.loc[overlaps, "red_flag_overlap"] = True

# Count the affected laps
print("Red-flag-overlapping laps:",
      development_laps["red_flag_overlap"].sum())

print("Races with red-flag-overlapping laps:",
      development_laps.loc[
          development_laps["red_flag_overlap"], "session_key"
      ].nunique())

Red-flag-overlapping laps: 20
Races with red-flag-overlapping laps: 2


In [96]:
# Flag laps overlapping either type of disruption
development_laps["race_control_disruption"] = (
    development_laps["sc_vsc_overlap"]
    | development_laps["red_flag_overlap"]
)

# Count unique affected and unaffected laps
print(
    "Disrupted laps:",
    development_laps["race_control_disruption"].sum()
)

print(
    "Unaffected laps:",
    (~development_laps["race_control_disruption"]).sum()
)

# Check how many laps overlap both categories
print(
    "Overlapping both SC/VSC and red flag:",
    (
        development_laps["sc_vsc_overlap"]
        & development_laps["red_flag_overlap"]
    ).sum()
)

Disrupted laps: 2473
Unaffected laps: 43577
Overlapping both SC/VSC and red flag: 18


In [97]:
# Identify development laps without recorded race-control overlap
clean_lap_indices = development_laps.index[
    ~development_laps["race_control_disruption"]
]

# Create filtered copies of our development features and target
X_dev_clean = X_dev.loc[clean_lap_indices].copy()
y_dev_clean = y_dev.loc[clean_lap_indices].copy()

# Check that features and target still match
print("Clean development features:", X_dev_clean.shape)
print("Clean development targets:", y_dev_clean.shape)
print("Original development laps:", len(X_dev))
print("Laps excluded:", len(X_dev) - len(X_dev_clean))

Clean development features: (43577, 9)
Clean development targets: (43577,)
Original development laps: 46050
Laps excluded: 2473


In [98]:
# Create a new Random Forest using our existing settings
rf_race_control_clean = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using only development laps without recorded disruptions
rf_race_control_clean.fit(
    X_dev_clean[tyre_features],
    y_dev_clean
)

# Predict the unchanged validation dataset
race_control_clean_predictions = rf_race_control_clean.predict(
    X_val[tyre_features]
)

# Compare the original and newly trained models
print(
    "Original validation MAE:",
    round(mean_absolute_error(y_val, tyres_only_predictions), 2)
)

print(
    "Race-control-cleaned validation MAE:",
    round(mean_absolute_error(y_val, race_control_clean_predictions), 2)
)

Original validation MAE: 9.89
Race-control-cleaned validation MAE: 7.61


In [99]:
# Compare errors for each tyre compound
compound_results = pd.DataFrame({
    "compound": X_val["compound"],
    "actual": y_val,
    "original_prediction": tyres_only_predictions,
    "cleaned_prediction": race_control_clean_predictions
})

# Calculate absolute prediction errors
compound_results["original_error"] = (
    compound_results["actual"]
    - compound_results["original_prediction"]
).abs()

compound_results["cleaned_error"] = (
    compound_results["actual"]
    - compound_results["cleaned_prediction"]
).abs()

# Display the average error for each compound
print(
    compound_results.groupby("compound")[
        ["original_error", "cleaned_error"]
    ].mean().round(2)
)

              original_error  cleaned_error
compound                                   
HARD                    6.78           6.27
INTERMEDIATE           21.28          10.40
MEDIUM                  8.35           7.47
SOFT                   10.34          10.74
WET                    27.51          30.32


In [100]:
# Count validation laps for each tyre compound
print(
    X_val["compound"]
    .value_counts()
    .to_string()
)

compound
HARD            5506
MEDIUM          4658
INTERMEDIATE    2069
SOFT             503
WET               28


## Linear Regression

In [101]:
# Import our second machine-learning model
from sklearn.linear_model import LinearRegression

# Build a pipeline using the same feature preprocessing
linear_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", LinearRegression())
])

# Train on the cleaned development dataset
linear_model.fit(
    X_dev_clean[tyre_features],
    y_dev_clean
)

# Predict lap times for the unchanged validation races
linear_predictions = linear_model.predict(
    X_val[tyre_features]
)

# Compare Linear Regression with our cleaned Random Forest
print(
    "Linear Regression MAE:",
    round(mean_absolute_error(y_val, linear_predictions), 2)
)

print(
    "Cleaned Random Forest MAE:",
    round(mean_absolute_error(y_val, race_control_clean_predictions), 2)
)

Linear Regression MAE: 5.01
Cleaned Random Forest MAE: 7.61


In [102]:
# Create a comparison table for each validation lap
linear_compound_results = pd.DataFrame({
    "compound": X_val["compound"],
    "actual": y_val,
    "linear_prediction": linear_predictions
})

# Calculate the absolute prediction error
linear_compound_results["error"] = (
    linear_compound_results["actual"]
    - linear_compound_results["linear_prediction"]
).abs()

# Show the average error for each tyre compound
print(
    linear_compound_results.groupby("compound")["error"]
    .mean()
    .round(2)
)

compound
HARD             4.28
INTERMEDIATE     6.00
MEDIUM           5.21
SOFT             5.60
WET             31.36
Name: error, dtype: float64


In [103]:
# Import two additional regression metrics
from sklearn.metrics import mean_squared_error, r2_score

# Calculate Linear Regression's validation metrics
linear_rmse = mean_squared_error(
    y_val, linear_predictions
) ** 0.5

linear_r2 = r2_score(
    y_val, linear_predictions
)

# Display the results
print("Linear Regression MAE:", round(mean_absolute_error(y_val, linear_predictions), 2))
print("Linear Regression RMSE:", round(linear_rmse, 2))
print("Linear Regression R²:", round(linear_r2, 3))

Linear Regression MAE: 5.01
Linear Regression RMSE: 10.39
Linear Regression R²: 0.425


In [104]:
# Import our third machine-learning model
from sklearn.ensemble import GradientBoostingRegressor

# Build Gradient Boosting with the same tyre and circuit features
gradient_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train on the cleaned development laps
gradient_model.fit(
    X_dev_clean[tyre_features],
    y_dev_clean
)

# Predict lap times for the unchanged validation races
gradient_predictions = gradient_model.predict(
    X_val[tyre_features]
)

# Compare all three models
print("Linear Regression MAE:", round(mean_absolute_error(y_val, linear_predictions), 2))
print("Random Forest MAE:", round(mean_absolute_error(y_val, race_control_clean_predictions), 2))
print("Gradient Boosting MAE:", round(mean_absolute_error(y_val, gradient_predictions), 2))

Linear Regression MAE: 5.01
Random Forest MAE: 7.61
Gradient Boosting MAE: 4.59


In [105]:
# Create a table of Gradient Boosting predictions
gradient_compound_results = pd.DataFrame({
    "compound": X_val["compound"],
    "actual": y_val,
    "prediction": gradient_predictions
})

# Calculate the absolute error for each lap
gradient_compound_results["error"] = (
    gradient_compound_results["actual"]
    - gradient_compound_results["prediction"]
).abs()

# Calculate average error by tyre compound
print(
    gradient_compound_results.groupby("compound")["error"]
    .mean()
    .round(2)
)

compound
HARD             3.99
INTERMEDIATE     5.08
MEDIUM           4.86
SOFT             5.17
WET             33.18
Name: error, dtype: float64


In [106]:
# Build Gradient Boosting with tyre, weather and circuit features
gradient_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train on cleaned development laps with all available features
gradient_weather_model.fit(
    X_dev_clean,
    y_dev_clean
)

# Predict using the unchanged validation dataset
gradient_weather_predictions = gradient_weather_model.predict(
    X_val
)

# Compare Gradient Boosting with and without weather
print(
    "Tyres + circuit MAE:",
    round(mean_absolute_error(y_val, gradient_predictions), 2)
)

print(
    "Tyres + weather + circuit MAE:",
    round(mean_absolute_error(y_val, gradient_weather_predictions), 2)
)

Tyres + circuit MAE: 4.59
Tyres + weather + circuit MAE: 8.32


In [107]:
# Get the feature names after preprocessing
weather_feature_names = (
    gradient_weather_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get the importance assigned to each feature
weather_importances = (
    gradient_weather_model.named_steps["regressor"]
    .feature_importances_
)

# Create a table and show the 12 most important features
importance_table = pd.DataFrame({
    "feature": weather_feature_names,
    "importance": weather_importances
})

print(
    importance_table.sort_values(
        "importance", ascending=False
    ).head(12).to_string(index=False)
)

                              feature  importance
           remainder__air_temperature    0.237583
categories__circuit_Spa-Francorchamps    0.224055
                  remainder__pressure    0.205912
         remainder__track_temperature    0.123112
                  remainder__humidity    0.043393
             categories__circuit_Baku    0.036560
            categories__circuit_Monza    0.024464
           categories__circuit_Austin    0.023894
                  remainder__tyre_age    0.016544
        categories__circuit_Barcelona    0.011229
    categories__compound_INTERMEDIATE    0.008065
         categories__circuit_Montréal    0.007602


In [108]:
# Compare both Gradient Boosting models on each validation race
weather_race_results = pd.DataFrame({
    "race": df.loc[X_val.index, "race_name"],
    "actual": y_val,
    "tyres_prediction": gradient_predictions,
    "weather_prediction": gradient_weather_predictions
})

# Calculate each model's absolute prediction error
weather_race_results["tyres_error"] = (
    weather_race_results["actual"]
    - weather_race_results["tyres_prediction"]
).abs()

weather_race_results["weather_error"] = (
    weather_race_results["actual"]
    - weather_race_results["weather_prediction"]
).abs()

# Compare average errors by race
print(
    weather_race_results.groupby("race")[
        ["tyres_error", "weather_error"]
    ].mean().round(2).sort_values(
        "weather_error", ascending=False
    ).to_string()
)

                tyres_error  weather_error
race                                      
Spielberg              3.32          22.59
Japan                  4.14          14.92
Baku                   3.23          11.79
Lusail                 5.66          11.03
Mexico                 6.69          10.13
Canada                 6.65           7.45
Brazil                 6.12           7.00
Imola                  8.26           6.94
United Kingdom         2.73           4.93
Netherlands            2.03           2.15
Monza                  2.46           1.71
Suzuka                 3.03           1.30


In [109]:
# Select Spielberg laps from development and validation
spielberg_dev = X_dev_clean[
    X_dev_clean["circuit"] == "Spielberg"
]

spielberg_val = X_val[
    X_val["circuit"] == "Spielberg"
]

# Compare average weather measurements
weather_columns = [
    "air_temperature",
    "track_temperature",
    "humidity",
    "pressure",
    "rainfall"
]

print("Development laps:", len(spielberg_dev))
print("Validation laps:", len(spielberg_val))

print("\nDevelopment weather averages:")
print(spielberg_dev[weather_columns].mean().round(2))

print("\nValidation weather averages:")
print(spielberg_val[weather_columns].mean().round(2))

Development laps: 1220
Validation laps: 1092

Development weather averages:
air_temperature       22.68
track_temperature     32.12
humidity              52.28
pressure             934.00
rainfall               0.00
dtype: float64

Validation weather averages:
air_temperature       30.74
track_temperature     48.84
humidity              31.15
pressure             943.01
rainfall               0.00
dtype: float64


In [110]:
# Select Spielberg validation laps
spielberg_results = weather_race_results[
    weather_race_results["race"] == "Spielberg"
]

# Compare actual lap times with both models' predictions
print(
    "Actual average lap time:",
    round(spielberg_results["actual"].mean(), 2)
)

print(
    "Tyres + circuit predicted average:",
    round(spielberg_results["tyres_prediction"].mean(), 2)
)

print(
    "With weather predicted average:",
    round(spielberg_results["weather_prediction"].mean(), 2)
)

Actual average lap time: 72.49
Tyres + circuit predicted average: 72.61
With weather predicted average: 94.08


In [111]:
# Compare Spielberg's training and validation temperature ranges
for column in ["air_temperature", "track_temperature"]:
    print("\n", column)

    # Show the lowest and highest training temperatures
    print(
        "Development range:",
        round(spielberg_dev[column].min(), 2),
        "to",
        round(spielberg_dev[column].max(), 2)
    )

    # Show the lowest and highest validation temperatures
    print(
        "Validation range:",
        round(spielberg_val[column].min(), 2),
        "to",
        round(spielberg_val[column].max(), 2)
    )


 air_temperature
Development range: 22.2 to 23.2
Validation range: 30.2 to 31.3

 track_temperature
Development range: 30.6 to 34.4
Validation range: 45.9 to 50.4


In [112]:
# Compare temperature ranges across all cleaned development laps
for column in ["air_temperature", "track_temperature"]:
    print("\n" + column)

    # Full training temperature range
    print(
        "All development races:",
        round(X_dev_clean[column].min(), 2),
        "to",
        round(X_dev_clean[column].max(), 2)
    )

    # Spielberg validation temperature range
    print(
        "Spielberg validation:",
        round(spielberg_val[column].min(), 2),
        "to",
        round(spielberg_val[column].max(), 2)
    )


air_temperature
All development races: 15.1 to 33.9
Spielberg validation: 30.2 to 31.3

track_temperature
All development races: 16.7 to 52.7
Spielberg validation: 45.9 to 50.4


In [113]:
# Copy Spielberg's validation features without changing the originals
spielberg_temperature_test = spielberg_val.copy()

# Replace temperatures with Spielberg development averages
spielberg_temperature_test["air_temperature"] = (
    spielberg_dev["air_temperature"].mean()
)

spielberg_temperature_test["track_temperature"] = (
    spielberg_dev["track_temperature"].mean()
)

# Predict with the original weather model
temperature_test_predictions = gradient_weather_model.predict(
    spielberg_temperature_test
)

# Compare average predictions before and after changing temperatures
print(
    "Original weather prediction:",
    round(spielberg_results["weather_prediction"].mean(), 2)
)

print(
    "Adjusted-temperature prediction:",
    round(temperature_test_predictions.mean(), 2)
)

print(
    "Actual average lap time:",
    round(spielberg_results["actual"].mean(), 2)
)

Original weather prediction: 94.08
Adjusted-temperature prediction: 89.16
Actual average lap time: 72.49


In [114]:
# Copy the original Spielberg validation features
spielberg_pressure_test = spielberg_val.copy()

# Replace only pressure with Spielberg's development average
spielberg_pressure_test["pressure"] = (
    spielberg_dev["pressure"].mean()
)

# Predict lap times using the weather model
pressure_test_predictions = gradient_weather_model.predict(
    spielberg_pressure_test
)

# Compare average predictions
print(
    "Original weather prediction:",
    round(spielberg_results["weather_prediction"].mean(), 2)
)

print(
    "Adjusted-pressure prediction:",
    round(pressure_test_predictions.mean(), 2)
)

print(
    "Actual average lap time:",
    round(spielberg_results["actual"].mean(), 2)
)

Original weather prediction: 94.08
Adjusted-pressure prediction: 83.35
Actual average lap time: 72.49


In [115]:
# Create feature lists without atmospheric pressure
features_without_pressure = [
    column for column in X_dev_clean.columns
    if column != "pressure"
]

# Use the existing preprocessing, which handles circuit and compound
gradient_no_pressure = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train without pressure
gradient_no_pressure.fit(
    X_dev_clean[features_without_pressure],
    y_dev_clean
)

# Predict the same validation races
no_pressure_predictions = gradient_no_pressure.predict(
    X_val[features_without_pressure]
)

# Compare the three models
print("Tyres + circuit:", round(mean_absolute_error(y_val, gradient_predictions), 2))
print("All weather features:", round(mean_absolute_error(y_val, gradient_weather_predictions), 2))
print("Weather without pressure:", round(mean_absolute_error(y_val, no_pressure_predictions), 2))

Tyres + circuit: 4.59
All weather features: 8.32
Weather without pressure: 7.52


In [116]:
# Select Spielberg laps from our existing validation results
spielberg_mask = (
    X_val["circuit"] == "Spielberg"
).to_numpy()

# Compare predictions for the same Spielberg laps
print(
    "Original weather model:",
    round(
        mean_absolute_error(
            y_val.iloc[spielberg_mask],
            gradient_weather_predictions[spielberg_mask]
        ), 2
    )
)

print(
    "Weather model without pressure:",
    round(
        mean_absolute_error(
            y_val.iloc[spielberg_mask],
            no_pressure_predictions[spielberg_mask]
        ), 2
    )
)

Original weather model: 22.59
Weather model without pressure: 13.38


In [117]:
# Import the race-grouped cross-validation method
from sklearn.model_selection import GroupKFold

# Count the unique races in our original outer training dataset
training_race_count = race_groups.iloc[train_idx].nunique()

# Prepare five-fold cross-validation
race_cv = GroupKFold(n_splits=5)

# Confirm our setup
print("Available training races:", training_race_count)
print("Cross-validation folds:", race_cv.n_splits)

Available training races: 56
Cross-validation folds: 5


In [118]:
# Get all race IDs in the outer training dataset
all_training_sessions = set(
    race_groups.iloc[train_idx].unique()
)

# Get race IDs already checked against race-control messages
checked_sessions = set(
    development_laps["session_key"].unique()
)

# Find races that still need race-control investigation
missing_sessions = sorted(
    all_training_sessions - checked_sessions
)

# Display the results
print("Training races:", len(all_training_sessions))
print("Already checked:", len(checked_sessions))
print("Still need checking:", len(missing_sessions))
print("Missing session keys:", missing_sessions)

Training races: 56
Already checked: 44
Still need checking: 12
Missing session keys: [9070, 9157, 9221, 9496, 9531, 9558, 9582, 9625, 9636, 9955, 9987, 10006]


In [119]:
# Download race-control messages for the 12 missing races
additional_records = []
failed_sessions = []

for session_key in missing_sessions:
    # Request messages for one race
    response = requests.get(
        "https://api.openf1.org/v1/race_control",
        params={"session_key": int(session_key)},
        timeout=30
    )

    # Record unsuccessful requests
    if response.status_code != 200:
        failed_sessions.append(session_key)
        continue

    # Add the race's messages to our collection
    additional_records.extend(response.json())

# Store the downloaded messages
additional_race_control = pd.DataFrame(additional_records)

# Check how much data we retrieved
print("Messages downloaded:", len(additional_race_control))
print("Races retrieved:", additional_race_control["session_key"].nunique())
print("Failed sessions:", failed_sessions)

Messages downloaded: 1051
Races retrieved: 12
Failed sessions: []


In [120]:
# Combine the original and newly downloaded race-control messages
all_training_race_control = pd.concat(
    [development_race_control, additional_race_control],
    ignore_index=True
)

# Check that all 56 training races are represented
print("Total messages:", len(all_training_race_control))
print(
    "Races covered:",
    all_training_race_control["session_key"].nunique()
)

# Check for duplicate messages
print(
    "Duplicate messages:",
    all_training_race_control.duplicated().sum()
)

Total messages: 5108
Races covered: 56
Duplicate messages: 0


In [121]:
# Copy all race-control messages into a new timeline
all_race_control_timeline = all_training_race_control.copy()

# Convert message timestamps to UTC datetimes
all_race_control_timeline["event_time"] = pd.to_datetime(
    all_race_control_timeline["date"],
    utc=True,
    format="mixed",
    errors="coerce"
)

# Sort messages by race and time
all_race_control_timeline = (
    all_race_control_timeline
    .sort_values(["session_key", "event_time"])
    .reset_index(drop=True)
)

# Check that every message has a valid timestamp
print("Total messages:", len(all_race_control_timeline))
print(
    "Valid timestamps:",
    all_race_control_timeline["event_time"].notna().sum()
)
print(
    "Missing timestamps:",
    all_race_control_timeline["event_time"].isna().sum()
)

Total messages: 5108
Valid timestamps: 5108
Missing timestamps: 0


In [122]:
# List the race-control messages that mark disruptions
disruption_messages = [
    "SAFETY CAR DEPLOYED",
    "SAFETY CAR IN THIS LAP",
    "VIRTUAL SAFETY CAR DEPLOYED",
    "VIRTUAL SAFETY CAR ENDING",
    "RED FLAG",
    "CHEQUERED FLAG"
]

# Select those messages from our combined timeline
all_status_events = all_race_control_timeline[
    all_race_control_timeline["message"].isin(disruption_messages)
].copy()

# Count each type of message
print(
    all_status_events["message"]
    .value_counts()
    .to_string()
)

# Check how many races contain these messages
print(
    "\nRaces represented:",
    all_status_events["session_key"].nunique()
)

message
CHEQUERED FLAG                 55
SAFETY CAR DEPLOYED            39
SAFETY CAR IN THIS LAP         33
VIRTUAL SAFETY CAR DEPLOYED    26
VIRTUAL SAFETY CAR ENDING      23
RED FLAG                       10

Races represented: 55


In [123]:
# Get every race in our combined race-control dataset
all_sessions = set(
    all_race_control_timeline["session_key"].unique()
)

# Get races containing at least one of our selected event messages
sessions_with_events = set(
    all_status_events["session_key"].unique()
)

# Find the race missing from our selected events
missing_event_sessions = sorted(
    all_sessions - sessions_with_events
)

print("Races without selected events:", missing_event_sessions)

# Show the messages recorded for that race
for session_key in missing_event_sessions:
    print("\nSession:", session_key)
    print(
        all_race_control_timeline.loc[
            all_race_control_timeline["session_key"] == session_key,
            "message"
        ].value_counts().head(15).to_string()
    )

Races without selected events: [9157]

Session: 9157
message
GREEN LIGHT - PIT EXIT OPEN                                                                                     2
PINK HEAD PADDING MATERIAL MUST BE USED                                                                         1
CAR 4 (NOR) TIME 1:27.567 DELETED - TRACK LIMITS AT TURN 2 LAP 38 16:18:11                                      1
WAVED BLUE FLAG FOR CAR 20 (MAG) TIMED AT 16:20:03                                                              1
WAVED BLUE FLAG FOR CAR 20 (MAG) TIMED AT 16:20:22                                                              1
WAVED BLUE FLAG FOR CAR 31 (OCO) TIMED AT 16:21:07                                                              1
CAR 81 (PIA) TIME 1:26.853 DELETED - TRACK LIMITS AT TURN 5 LAP 40 16:20:57                                     1
CAR 44 (HAM) TIME 1:26.658 DELETED - TRACK LIMITS AT TURN 5 LAP 40 16:20:58                                     1
TURN 4 INCIDENT INVOLVING C

In [124]:
# Select every race-control message from session 9157
session_9157 = all_race_control_timeline[
    all_race_control_timeline["session_key"] == 9157
]

# Search for possible disruption messages with different wording
keywords = r"SAFETY CAR|VSC|RED FLAG|CHEQUERED|CHECKERED"

matching_messages = session_9157[
    session_9157["message"].str.contains(
        keywords,
        case=False,
        na=False
    )
]

# Display any matches and the total number of messages
print("Total session messages:", len(session_9157))
print("Keyword matches:", len(matching_messages))
print(matching_messages[["date", "message"]].to_string(index=False))

Total session messages: 45
Keyword matches: 0
Empty DataFrame
Columns: [date, message]
Index: []


In [125]:
# Select race-control events that mark disruptions or race endings
status_events_all = all_race_control_timeline[
    all_race_control_timeline["message"].isin([
        "SAFETY CAR DEPLOYED",
        "SAFETY CAR IN THIS LAP",
        "VIRTUAL SAFETY CAR DEPLOYED",
        "VIRTUAL SAFETY CAR ENDING",
        "RED FLAG",
        "CHEQUERED FLAG"
    ])
].copy()

# Keep events in chronological order within each race
status_events_all = status_events_all.sort_values(
    ["session_key", "event_time"]
)

# Check the event table
print("Total status events:", len(status_events_all))
print("Races with status events:", status_events_all["session_key"].nunique())
print("Races without status events:", 56 - status_events_all["session_key"].nunique())

Total status events: 186
Races with status events: 55
Races without status events: 1


In [126]:
# Select Safety Car and Virtual Safety Car deployment events
deployment_events = status_events_all[
    status_events_all["message"].isin([
        "SAFETY CAR DEPLOYED",
        "VIRTUAL SAFETY CAR DEPLOYED"
    ])
].copy()

# Count each type of deployment
print(
    deployment_events["message"]
    .value_counts()
    .to_string()
)

# Count races with at least one deployment
print(
    "\nRaces with SC/VSC deployments:",
    deployment_events["session_key"].nunique()
)

message
SAFETY CAR DEPLOYED            39
VIRTUAL SAFETY CAR DEPLOYED    26

Races with SC/VSC deployments: 35


In [127]:
# Store the interruption intervals for all training races
all_disruption_intervals = []

# Process each race separately
for session, events in status_events_all.groupby("session_key"):
    active_type = None
    start_time = None

    for _, event in events.iterrows():
        message = event["message"].strip()
        event_time = event["event_time"]

        # Start a Safety Car or VSC period
        if message in [
            "SAFETY CAR DEPLOYED",
            "VIRTUAL SAFETY CAR DEPLOYED"
        ]:
            # Close an existing period if another deployment replaces it
            if active_type is not None:
                all_disruption_intervals.append({
                    "session_key": session,
                    "type": active_type,
                    "start": start_time,
                    "end": event_time
                })

            active_type = (
                "SC" if message == "SAFETY CAR DEPLOYED" else "VSC"
            )
            start_time = event_time

        # End a period at its matching ending or a race stoppage
        elif active_type is not None:
            matching_end = (
                (active_type == "SC" and message == "SAFETY CAR IN THIS LAP")
                or
                (active_type == "VSC" and message == "VIRTUAL SAFETY CAR ENDING")
            )

            if matching_end or message in ["RED FLAG", "CHEQUERED FLAG"]:
                all_disruption_intervals.append({
                    "session_key": session,
                    "type": active_type,
                    "start": start_time,
                    "end": event_time
                })

                active_type = None
                start_time = None

# Convert the intervals into a DataFrame
all_disruption_intervals = pd.DataFrame(all_disruption_intervals)

# Inspect the results
print("Total intervals:", len(all_disruption_intervals))
print(all_disruption_intervals["type"].value_counts())
print(
    "Races with intervals:",
    all_disruption_intervals["session_key"].nunique()
)

Total intervals: 65
type
SC     39
VSC    26
Name: count, dtype: int64
Races with intervals: 35


In [128]:
# Find races where a deployment has no matching closing event
unclosed_sessions = []

# Check the final active disruption state in each race
for session, events in status_events_all.groupby("session_key"):
    active_type = None

    for _, event in events.iterrows():
        message = event["message"].strip()

        # Record a new deployment
        if message == "SAFETY CAR DEPLOYED":
            active_type = "SC"
        elif message == "VIRTUAL SAFETY CAR DEPLOYED":
            active_type = "VSC"

        # Close a Safety Car period
        elif message == "SAFETY CAR IN THIS LAP" and active_type == "SC":
            active_type = None

        # Close a Virtual Safety Car period
        elif message == "VIRTUAL SAFETY CAR ENDING" and active_type == "VSC":
            active_type = None

        # Close either type at a red flag or race finish
        elif message in ["RED FLAG", "CHEQUERED FLAG"]:
            active_type = None

    # Record races with an unfinished disruption
    if active_type is not None:
        unclosed_sessions.append((session, active_type))

print("Unclosed intervals:", len(unclosed_sessions))
print("Affected sessions:", unclosed_sessions)

Unclosed intervals: 0
Affected sessions: []


In [129]:
# Select red-flag messages from all 56 training races
all_red_flag_events = all_race_control_timeline[
    all_race_control_timeline["message"] == "RED FLAG"
].copy()

# Count red flags and affected races
print("Red-flag messages:", len(all_red_flag_events))
print(
    "Races with red flags:",
    all_red_flag_events["session_key"].nunique()
)

# Show the affected sessions
print(
    "Session keys:",
    sorted(all_red_flag_events["session_key"].unique())
)

Red-flag messages: 10
Races with red flags: 7
Session keys: [7787, 9181, 9205, 9496, 9523, 9636, 9939]


In [130]:
# Find session restart messages across all 56 races
all_session_starts = all_race_control_timeline[
    all_race_control_timeline["message"] == "SESSION STARTED"
].copy()

# Count restart messages
print("Session started messages:", len(all_session_starts))

# Check which red-flag races have restart messages
red_flag_sessions = set(all_red_flag_events["session_key"])
restart_sessions = set(all_session_starts["session_key"])

print(
    "Red-flag races with restart messages:",
    len(red_flag_sessions & restart_sessions)
)

print(
    "Red-flag races without restart messages:",
    sorted(red_flag_sessions - restart_sessions)
)

Session started messages: 64
Red-flag races with restart messages: 7
Red-flag races without restart messages: []


In [131]:
# Store red-flag intervals for all training races
all_red_flag_intervals = []

# Process each race with recorded red flags
for session, flags in all_red_flag_events.groupby("session_key"):

    # Find the race's chequered flag, if recorded
    finish_time = status_events_all.loc[
        (status_events_all["session_key"] == session)
        & (status_events_all["message"] == "CHEQUERED FLAG"),
        "event_time"
    ].min()

    # Find restart messages for this race
    starts = all_session_starts.loc[
        all_session_starts["session_key"] == session,
        "event_time"
    ]

    for _, flag in flags.iterrows():
        flag_time = flag["event_time"]

        # Ignore red flags recorded after the race finished
        if pd.notna(finish_time) and flag_time >= finish_time:
            continue

        # Find the first restart after this red flag
        later_starts = starts[starts > flag_time]

        if not later_starts.empty:
            all_red_flag_intervals.append({
                "session_key": session,
                "start": flag_time,
                "end": later_starts.min()
            })

# Convert the results into a DataFrame
all_red_flag_intervals = pd.DataFrame(all_red_flag_intervals)

# Check the expanded intervals
print("Red-flag intervals:", len(all_red_flag_intervals))
print("Races represented:", all_red_flag_intervals["session_key"].nunique())
print(all_red_flag_intervals.to_string(index=False))

Red-flag intervals: 9
Races represented: 7
 session_key                     start                              end
        7787 2023-04-02 05:17:02+00:00 2023-04-02 05:33:00.429000+00:00
        7787 2023-04-02 06:41:07+00:00 2023-04-02 06:56:00.419000+00:00
        7787 2023-04-02 07:00:26+00:00 2023-04-02 07:33:00.456000+00:00
        9181 2023-10-29 20:51:53+00:00 2023-10-29 21:13:00.310000+00:00
        9205 2023-11-05 17:06:35+00:00 2023-11-05 17:31:00.366000+00:00
        9496 2024-04-07 05:04:24+00:00 2024-04-07 05:32:00.407000+00:00
        9523 2024-05-26 13:04:08+00:00 2024-05-26 13:44:00.121000+00:00
        9636 2024-11-03 16:37:22+00:00 2024-11-03 17:02:00.390000+00:00
        9939 2025-07-27 13:01:58+00:00 2025-07-27 14:20:00.065000+00:00


In [132]:
# Calculate the duration of each red-flag stoppage
all_red_flag_intervals["duration_minutes"] = (
    all_red_flag_intervals["end"]
    - all_red_flag_intervals["start"]
).dt.total_seconds() / 60

# Check for invalid or unusually long intervals
print(
    "Invalid intervals:",
    (all_red_flag_intervals["duration_minutes"] <= 0).sum()
)

print(
    "Shortest interval (minutes):",
    round(all_red_flag_intervals["duration_minutes"].min(), 2)
)

print(
    "Longest interval (minutes):",
    round(all_red_flag_intervals["duration_minutes"].max(), 2)
)

Invalid intervals: 0
Shortest interval (minutes): 14.89
Longest interval (minutes): 78.03


In [133]:
# Copy all laps belonging to our 56 outer-training races
all_training_laps = df.loc[X_train_circuit.index].copy()

# Convert each lap's starting timestamp to UTC
all_training_laps["lap_start_utc"] = pd.to_datetime(
    all_training_laps["date_start"],
    utc=True,
    format="mixed",
    errors="coerce"
)

# Calculate each lap's finishing timestamp
all_training_laps["lap_end_utc"] = (
    all_training_laps["lap_start_utc"]
    + pd.to_timedelta(all_training_laps["lap_duration"], unit="s")
)

# Check that all laps have usable timestamps
print("Training laps:", len(all_training_laps))
print("Training races:", all_training_laps["session_key"].nunique())
print("Missing start times:", all_training_laps["lap_start_utc"].isna().sum())
print("Missing end times:", all_training_laps["lap_end_utc"].isna().sum())

Training laps: 58814
Training races: 56
Missing start times: 0
Missing end times: 0


In [134]:
# Start by assuming no laps overlap an SC/VSC interval
all_training_laps["sc_vsc_overlap"] = False

# Check each race's interruption intervals
for session, intervals in all_disruption_intervals.groupby("session_key"):

    # Select laps from this race
    race_laps = all_training_laps["session_key"] == session

    for _, interval in intervals.iterrows():

        # Flag laps that overlap the interruption
        overlap = (
            race_laps
            & (all_training_laps["lap_start_utc"] < interval["end"])
            & (all_training_laps["lap_end_utc"] > interval["start"])
        )

        all_training_laps.loc[overlap, "sc_vsc_overlap"] = True

# Check how many laps were flagged
print("SC/VSC-overlapping laps:", all_training_laps["sc_vsc_overlap"].sum())
print(
    "Races with flagged laps:",
    all_training_laps.loc[
        all_training_laps["sc_vsc_overlap"], "session_key"
    ].nunique()
)

SC/VSC-overlapping laps: 3193
Races with flagged laps: 35


In [135]:
# Start by assuming no laps overlap a red-flag stoppage
all_training_laps["red_flag_overlap"] = False

# Check the red-flag intervals race by race
for session, intervals in all_red_flag_intervals.groupby("session_key"):

    # Select laps belonging to this race
    race_laps = all_training_laps["session_key"] == session

    for _, interval in intervals.iterrows():

        # Flag laps that overlap the red-flag interval
        overlap = (
            race_laps
            & (all_training_laps["lap_start_utc"] < interval["end"])
            & (all_training_laps["lap_end_utc"] > interval["start"])
        )

        all_training_laps.loc[overlap, "red_flag_overlap"] = True

# Check the results
print(
    "Red-flag-overlapping laps:",
    all_training_laps["red_flag_overlap"].sum()
)

print(
    "Races with flagged laps:",
    all_training_laps.loc[
        all_training_laps["red_flag_overlap"], "session_key"
    ].nunique()
)

Red-flag-overlapping laps: 36
Races with flagged laps: 3


In [136]:
# Combine both race-control disruption flags
all_training_laps["race_control_disruption"] = (
    all_training_laps["sc_vsc_overlap"]
    | all_training_laps["red_flag_overlap"]
)

# Check how many laps were flagged by both methods
both_flags = (
    all_training_laps["sc_vsc_overlap"]
    & all_training_laps["red_flag_overlap"]
)

# Count unique disrupted laps and remaining laps
print("Laps flagged by both:", both_flags.sum())
print(
    "Total disrupted laps:",
    all_training_laps["race_control_disruption"].sum()
)
print(
    "Laps without flagged disruptions:",
    (~all_training_laps["race_control_disruption"]).sum()
)

Laps flagged by both: 34
Total disrupted laps: 3195
Laps without flagged disruptions: 55619


In [137]:
# Find training laps from the race with uncertain race-control coverage
unknown_control_laps = all_training_laps[
    all_training_laps["session_key"] == 9157
]

# Check how many laps are affected
print("Session 9157 laps:", len(unknown_control_laps))

# Check whether any were flagged as disrupted
print(
    "Flagged disruptions:",
    unknown_control_laps["race_control_disruption"].sum()
)


Session 9157 laps: 923
Flagged disruptions: 0


In [138]:
# Mark the race whose disruption coverage is uncertain
all_training_laps["race_control_unknown"] = (
    all_training_laps["session_key"] == 9157
)

# Count laps eligible for clean training
eligible_training_laps = (
    ~all_training_laps["race_control_disruption"]
    & ~all_training_laps["race_control_unknown"]
)

print("Unknown-status laps:", all_training_laps["race_control_unknown"].sum())
print("Eligible training laps:", eligible_training_laps.sum())
print("Total training laps:", len(all_training_laps))

Unknown-status laps: 923
Eligible training laps: 54696
Total training laps: 58814


In [139]:
# Import the tool for cross-validation by race
from sklearn.model_selection import GroupKFold

# Create five folds, keeping each race together
race_cv = GroupKFold(n_splits=5)

# Generate the fold indices using our 56 training races
cv_splits = list(
    race_cv.split(
        X_train_circuit,
        y_train,
        groups=training_groups
    )
)

# Check how many folds we created
print("Number of folds:", len(cv_splits))

# Check that the first fold keeps training and validation races separate
fold_train_idx, fold_val_idx = cv_splits[0]

fold_train_races = set(training_groups.iloc[fold_train_idx])
fold_val_races = set(training_groups.iloc[fold_val_idx])

print("First fold training races:", len(fold_train_races))
print("First fold validation races:", len(fold_val_races))
print("Overlapping races:", len(fold_train_races & fold_val_races))

Number of folds: 5
First fold training races: 45
First fold validation races: 11
Overlapping races: 0


In [140]:
# Get the training and validation indices for fold 1
fold_train_idx, fold_val_idx = cv_splits[0]

# Find the lap indices belonging to fold 1's training races
fold_train_lap_indices = X_train_circuit.iloc[fold_train_idx].index

# Check which training laps are eligible
fold_train_eligible = eligible_training_laps.loc[
    fold_train_lap_indices
]

# Count the laps before and after filtering
print("Training laps before filtering:", len(fold_train_lap_indices))
print("Training laps after filtering:", fold_train_eligible.sum())
print("Training laps excluded:", (~fold_train_eligible).sum())

# Validation laps stay untouched
print("Validation laps:", len(fold_val_idx))

Training laps before filtering: 47155
Training laps after filtering: 43482
Training laps excluded: 3673
Validation laps: 11659


In [141]:
# Get the lap indices for fold 1
fold_train_lap_indices = X_train_circuit.iloc[fold_train_idx].index

# Keep only eligible laps from the training portion
clean_fold_train_indices = fold_train_lap_indices[
    eligible_training_laps.loc[fold_train_lap_indices].to_numpy()
]

# Prepare clean training data
X_fold_train = X_train_circuit.loc[clean_fold_train_indices].copy()
y_fold_train = y_train.loc[clean_fold_train_indices].copy()

# Prepare untouched validation data
X_fold_val = X_train_circuit.iloc[fold_val_idx].copy()
y_fold_val = y_train.iloc[fold_val_idx].copy()

# Check the final dataset sizes
print("X training rows:", len(X_fold_train))
print("y training rows:", len(y_fold_train))
print("X validation rows:", len(X_fold_val))
print("y validation rows:", len(y_fold_val))

X training rows: 43482
y training rows: 43482
X validation rows: 11659
y validation rows: 11659


In [142]:
# Create a fresh Gradient Boosting model for fold 1
fold1_gradient_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train using only the eligible training laps
fold1_gradient_model.fit(
    X_fold_train[tyre_features],
    y_fold_train
)

# Predict lap times for the untouched validation races
fold1_predictions = fold1_gradient_model.predict(
    X_fold_val[tyre_features]
)

# Measure prediction error in seconds
fold1_mae = mean_absolute_error(y_fold_val, fold1_predictions)

print("Fold 1 Gradient Boosting MAE:", round(fold1_mae, 2), "seconds")

Fold 1 Gradient Boosting MAE: 4.64 seconds


In [143]:
# Get the training and validation positions for fold 2
fold2_train_idx, fold2_val_idx = cv_splits[1]

# Find fold 2's training lap indices
fold2_train_lap_indices = X_train_circuit.iloc[fold2_train_idx].index

# Keep only eligible training laps
fold2_clean_indices = fold2_train_lap_indices[
    eligible_training_laps.loc[fold2_train_lap_indices].to_numpy()
]

# Prepare training and untouched validation data
X_fold2_train = X_train_circuit.loc[fold2_clean_indices]
y_fold2_train = y_train.loc[fold2_clean_indices]

X_fold2_val = X_train_circuit.iloc[fold2_val_idx]
y_fold2_val = y_train.iloc[fold2_val_idx]

# Create a fresh model with identical settings
fold2_gradient_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train and predict
fold2_gradient_model.fit(X_fold2_train[tyre_features], y_fold2_train)
fold2_predictions = fold2_gradient_model.predict(X_fold2_val[tyre_features])

# Calculate validation error
fold2_mae = mean_absolute_error(y_fold2_val, fold2_predictions)

print("Fold 2 Gradient Boosting MAE:", round(fold2_mae, 2), "seconds")

Fold 2 Gradient Boosting MAE: 7.68 seconds


In [144]:
# Get fold 3's training and validation positions
fold3_train_idx, fold3_val_idx = cv_splits[2]

# Keep only eligible training laps
fold3_train_indices = X_train_circuit.iloc[fold3_train_idx].index
fold3_clean_indices = fold3_train_indices[
    eligible_training_laps.loc[fold3_train_indices].to_numpy()
]

# Prepare the training and validation datasets
X_fold3_train = X_train_circuit.loc[fold3_clean_indices]
y_fold3_train = y_train.loc[fold3_clean_indices]
X_fold3_val = X_train_circuit.iloc[fold3_val_idx]
y_fold3_val = y_train.iloc[fold3_val_idx]

# Create a fresh copy of the same model
from sklearn.base import clone
fold3_gradient_model = clone(fold2_gradient_model)

# Train, predict, and evaluate
fold3_gradient_model.fit(X_fold3_train[tyre_features], y_fold3_train)
fold3_predictions = fold3_gradient_model.predict(X_fold3_val[tyre_features])

fold3_mae = mean_absolute_error(y_fold3_val, fold3_predictions)
print("Fold 3 Gradient Boosting MAE:", round(fold3_mae, 2), "seconds")

Fold 3 Gradient Boosting MAE: 6.32 seconds


In [145]:
# Get fold 4's training and validation positions
fold4_train_idx, fold4_val_idx = cv_splits[3]

# Keep only eligible training laps
fold4_train_indices = X_train_circuit.iloc[fold4_train_idx].index
fold4_clean_indices = fold4_train_indices[
    eligible_training_laps.loc[fold4_train_indices].to_numpy()
]

# Prepare training and untouched validation data
X_fold4_train = X_train_circuit.loc[fold4_clean_indices]
y_fold4_train = y_train.loc[fold4_clean_indices]

X_fold4_val = X_train_circuit.iloc[fold4_val_idx]
y_fold4_val = y_train.iloc[fold4_val_idx]

# Create a fresh model with the same settings
fold4_gradient_model = clone(fold3_gradient_model)

# Train the model and predict lap times
fold4_gradient_model.fit(X_fold4_train[tyre_features], y_fold4_train)
fold4_predictions = fold4_gradient_model.predict(X_fold4_val[tyre_features])

# Calculate the prediction error
fold4_mae = mean_absolute_error(y_fold4_val, fold4_predictions)

print("Fold 4 Gradient Boosting MAE:", round(fold4_mae, 2), "seconds")

Fold 4 Gradient Boosting MAE: 5.92 seconds


In [146]:
# Get fold 5's training and validation positions
fold5_train_idx, fold5_val_idx = cv_splits[4]

# Find eligible training laps
fold5_train_indices = X_train_circuit.iloc[fold5_train_idx].index
fold5_clean_indices = fold5_train_indices[
    eligible_training_laps.loc[fold5_train_indices].to_numpy()
]

# Prepare clean training data
X_fold5_train = X_train_circuit.loc[fold5_clean_indices]
y_fold5_train = y_train.loc[fold5_clean_indices]

# Prepare untouched validation data
X_fold5_val = X_train_circuit.iloc[fold5_val_idx]
y_fold5_val = y_train.iloc[fold5_val_idx]

# Create a fresh copy of the same model
fold5_gradient_model = clone(fold4_gradient_model)

# Train and predict
fold5_gradient_model.fit(
    X_fold5_train[tyre_features],
    y_fold5_train
)

fold5_predictions = fold5_gradient_model.predict(
    X_fold5_val[tyre_features]
)

# Calculate fold 5's prediction error
fold5_mae = mean_absolute_error(y_fold5_val, fold5_predictions)

print("Fold 5 Gradient Boosting MAE:", round(fold5_mae, 2), "seconds")

Fold 5 Gradient Boosting MAE: 7.45 seconds


In [147]:
 # Collect the MAE scores from all five folds
gradient_cv_scores = [
    fold1_mae,
    fold2_mae,
    fold3_mae,
    fold4_mae,
    fold5_mae
]

# Calculate the average and variation across folds
import numpy as np

print("Mean MAE:", round(np.mean(gradient_cv_scores), 2), "seconds")
print("Standard deviation:", round(np.std(gradient_cv_scores), 2), "seconds")
print("Best fold:", round(np.min(gradient_cv_scores), 2), "seconds")
print("Worst fold:", round(np.max(gradient_cv_scores), 2), "seconds")

Mean MAE: 6.4 seconds
Standard deviation: 1.1 seconds
Best fold: 4.64 seconds
Worst fold: 7.68 seconds


In [148]:
# Store our five-fold Gradient Boosting results
gradient_cv_summary = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "mae_seconds": gradient_cv_scores
})

# Display the results with a summary
print(gradient_cv_summary.round(2))
print("Mean MAE:", round(gradient_cv_summary["mae_seconds"].mean(), 2))
print("Standard deviation:", round(gradient_cv_summary["mae_seconds"].std(ddof=0), 2))

   fold  mae_seconds
0     1         4.64
1     2         7.68
2     3         6.32
3     4         5.92
4     5         7.45
Mean MAE: 6.4
Standard deviation: 1.1


In [149]:
# Create a fresh Linear Regression model
fold1_linear_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", LinearRegression())
])

# Train on the same clean laps used by Gradient Boosting
fold1_linear_model.fit(
    X_fold_train[tyre_features],
    y_fold_train
)

# Predict the same untouched validation laps
fold1_linear_predictions = fold1_linear_model.predict(
    X_fold_val[tyre_features]
)

# Calculate the prediction error
fold1_linear_mae = mean_absolute_error(
    y_fold_val,
    fold1_linear_predictions
)

print("Fold 1 Linear Regression MAE:", round(fold1_linear_mae, 2), "seconds")

Fold 1 Linear Regression MAE: 4.29 seconds


In [150]:
# Create a fresh Linear Regression model for fold 2
fold2_linear_model = clone(fold1_linear_model)

# Train on fold 2's eligible training laps
fold2_linear_model.fit(
    X_fold2_train[tyre_features],
    y_fold2_train
)

# Predict the untouched validation laps
fold2_linear_predictions = fold2_linear_model.predict(
    X_fold2_val[tyre_features]
)

# Calculate the prediction error
fold2_linear_mae = mean_absolute_error(
    y_fold2_val,
    fold2_linear_predictions
)

print("Fold 2 Linear Regression MAE:", round(fold2_linear_mae, 2), "seconds")

Fold 2 Linear Regression MAE: 6.31 seconds


In [151]:
# Create a fresh Linear Regression model for fold 3
fold3_linear_model = clone(fold1_linear_model)

# Train on fold 3's eligible training laps
fold3_linear_model.fit(
    X_fold3_train[tyre_features],
    y_fold3_train
)

# Predict the untouched validation laps
fold3_linear_predictions = fold3_linear_model.predict(
    X_fold3_val[tyre_features]
)

# Calculate the prediction error
fold3_linear_mae = mean_absolute_error(
    y_fold3_val,
    fold3_linear_predictions
)

print("Fold 3 Linear Regression MAE:", round(fold3_linear_mae, 2), "seconds")

Fold 3 Linear Regression MAE: 5.39 seconds


In [152]:
# Create a fresh Linear Regression model for fold 4
fold4_linear_model = clone(fold1_linear_model)

# Train using fold 4's eligible training laps
fold4_linear_model.fit(
    X_fold4_train[tyre_features],
    y_fold4_train
)

# Predict the untouched validation laps
fold4_linear_predictions = fold4_linear_model.predict(
    X_fold4_val[tyre_features]
)

# Calculate the prediction error
fold4_linear_mae = mean_absolute_error(
    y_fold4_val,
    fold4_linear_predictions
)

print("Fold 4 Linear Regression MAE:", round(fold4_linear_mae, 2), "seconds")

Fold 4 Linear Regression MAE: 5.3 seconds


In [153]:
# Create a fresh Linear Regression model for fold 5
fold5_linear_model = clone(fold1_linear_model)

# Train using fold 5's eligible training laps
fold5_linear_model.fit(
    X_fold5_train[tyre_features],
    y_fold5_train
)

# Predict the untouched validation laps
fold5_linear_predictions = fold5_linear_model.predict(
    X_fold5_val[tyre_features]
)

# Calculate the prediction error
fold5_linear_mae = mean_absolute_error(
    y_fold5_val,
    fold5_linear_predictions
)

print("Fold 5 Linear Regression MAE:", round(fold5_linear_mae, 2), "seconds")

Fold 5 Linear Regression MAE: 7.26 seconds


In [154]:
# Collect Linear Regression's five validation scores
linear_cv_scores = [
    fold1_linear_mae,
    fold2_linear_mae,
    fold3_linear_mae,
    fold4_linear_mae,
    fold5_linear_mae
]

# Calculate the average error and variation
print("Linear Regression mean MAE:",
      round(np.mean(linear_cv_scores), 2), "seconds")

print("Linear Regression standard deviation:",
      round(np.std(linear_cv_scores), 2), "seconds")

Linear Regression mean MAE: 5.71 seconds
Linear Regression standard deviation: 1.0 seconds


In [155]:
# Create a fresh Random Forest model
fold1_rf_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using the same eligible training laps
fold1_rf_model.fit(
    X_fold_train[tyre_features],
    y_fold_train
)

# Predict the same untouched validation laps
fold1_rf_predictions = fold1_rf_model.predict(
    X_fold_val[tyre_features]
)

# Calculate the prediction error
fold1_rf_mae = mean_absolute_error(
    y_fold_val,
    fold1_rf_predictions
)

print("Fold 1 Random Forest MAE:", round(fold1_rf_mae, 2), "seconds")

Fold 1 Random Forest MAE: 7.09 seconds


In [156]:
# Create a fresh Random Forest model for fold 2
fold2_rf_model = clone(fold1_rf_model)

# Train on fold 2's eligible training laps
fold2_rf_model.fit(
    X_fold2_train[tyre_features],
    y_fold2_train
)

# Predict the untouched validation laps
fold2_rf_predictions = fold2_rf_model.predict(
    X_fold2_val[tyre_features]
)

# Calculate the prediction error
fold2_rf_mae = mean_absolute_error(
    y_fold2_val,
    fold2_rf_predictions
)

print("Fold 2 Random Forest MAE:", round(fold2_rf_mae, 2), "seconds")

Fold 2 Random Forest MAE: 8.66 seconds


In [157]:
# Create a fresh Random Forest model for fold 3
fold3_rf_model = clone(fold1_rf_model)

# Train using fold 3's eligible training laps
fold3_rf_model.fit(
    X_fold3_train[tyre_features],
    y_fold3_train
)

# Predict lap times for the untouched validation races
fold3_rf_predictions = fold3_rf_model.predict(
    X_fold3_val[tyre_features]
)

# Calculate the mean absolute error
fold3_rf_mae = mean_absolute_error(
    y_fold3_val,
    fold3_rf_predictions
)

print("Fold 3 Random Forest MAE:", round(fold3_rf_mae, 2), "seconds")

Fold 3 Random Forest MAE: 6.38 seconds


In [158]:
# Create a fresh Random Forest model for fold 4
fold4_rf_model = clone(fold1_rf_model)

# Train using fold 4's eligible training laps
fold4_rf_model.fit(
    X_fold4_train[tyre_features],
    y_fold4_train
)

# Predict the untouched validation laps
fold4_rf_predictions = fold4_rf_model.predict(
    X_fold4_val[tyre_features]
)

# Calculate the prediction error
fold4_rf_mae = mean_absolute_error(
    y_fold4_val,
    fold4_rf_predictions
)

print("Fold 4 Random Forest MAE:", round(fold4_rf_mae, 2), "seconds")

Fold 4 Random Forest MAE: 7.77 seconds


In [159]:
# Create a fresh Random Forest model for fold 5
fold5_rf_model = clone(fold1_rf_model)

# Train on fold 5's eligible training laps
fold5_rf_model.fit(
    X_fold5_train[tyre_features],
    y_fold5_train
)

# Predict the untouched validation laps
fold5_rf_predictions = fold5_rf_model.predict(
    X_fold5_val[tyre_features]
)

# Calculate the mean absolute error
fold5_rf_mae = mean_absolute_error(
    y_fold5_val,
    fold5_rf_predictions
)

print("Fold 5 Random Forest MAE:", round(fold5_rf_mae, 2), "seconds")

Fold 5 Random Forest MAE: 8.54 seconds


In [160]:
# Collect Random Forest's five cross-validation scores
rf_cv_scores = [
    fold1_rf_mae,
    fold2_rf_mae,
    fold3_rf_mae,
    fold4_rf_mae,
    fold5_rf_mae
]

# Calculate its average error and variation
print("Random Forest mean MAE:",
      round(np.mean(rf_cv_scores), 2), "seconds")

print("Random Forest standard deviation:",
      round(np.std(rf_cv_scores), 2), "seconds")

Random Forest mean MAE: 7.69 seconds
Random Forest standard deviation: 0.87 seconds


In [161]:
# Combine the cross-validation scores for all three models
model_comparison = pd.DataFrame({
    "Linear Regression": linear_cv_scores,
    "Gradient Boosting": gradient_cv_scores,
    "Random Forest": rf_cv_scores
})

# Label the five race-grouped folds
model_comparison.index = [1, 2, 3, 4, 5]
model_comparison.index.name = "Fold"

# Display the results
print(model_comparison.round(2))

      Linear Regression  Gradient Boosting  Random Forest
Fold                                                     
1                  4.29               4.64           7.09
2                  6.31               7.68           8.66
3                  5.39               6.32           6.38
4                  5.30               5.92           7.77
5                  7.26               7.45           8.54


In [162]:
# Save all 15 cross-validation results to a CSV file
model_comparison.to_csv("f1_model_cv_results.csv")

# Confirm that the file was saved
print("Saved: f1_model_cv_results.csv")
print("Rows:", len(model_comparison))
print("Models:", len(model_comparison.columns))

Saved: f1_model_cv_results.csv
Rows: 5
Models: 3


In [163]:
# Check that our five race-grouped validation splits are available
print("Cross-validation folds:", len(cv_splits))

# Check that our training data includes the weather variables
print("\nAvailable training features:")
print(X_train_circuit.columns.tolist())

# Check how many training laps have eligible race-control status
print("\nEligible training laps:", eligible_training_laps.sum())

Cross-validation folds: 5

Available training features:
['compound', 'tyre_age', 'air_temperature', 'track_temperature', 'humidity', 'rainfall', 'wind_speed', 'pressure', 'circuit']

Eligible training laps: 54696


In [164]:
# Import the tools needed for our weather-inclusive model
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Select the first race-grouped cross-validation fold
weather_train_idx, weather_val_idx = cv_splits[0]

# Separate training and validation observations
X_weather_train = X_train_circuit.iloc[weather_train_idx].copy()
y_weather_train = y_train.iloc[weather_train_idx].copy()

X_weather_val = X_train_circuit.iloc[weather_val_idx].copy()
y_weather_val = y_train.iloc[weather_val_idx].copy()

# Remove disrupted or unknown-status laps from training only
weather_train_eligible = eligible_training_laps.loc[X_weather_train.index]

X_weather_train = X_weather_train.loc[weather_train_eligible].copy()
y_weather_train = y_weather_train.loc[weather_train_eligible].copy()

# Create the weather-inclusive Gradient Boosting model
weather_cv_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train using tyre, circuit and weather information
weather_cv_model.fit(X_weather_train, y_weather_train)

# Predict lap durations for the held-out races
weather_val_predictions = weather_cv_model.predict(X_weather_val)

# Calculate the prediction error
weather_fold1_mae = mean_absolute_error(
    y_weather_val,
    weather_val_predictions
)

# Display the first-fold result
print("Weather-inclusive Gradient Boosting — Fold 1")
print("Training laps:", len(X_weather_train))
print("Validation laps:", len(X_weather_val))
print("MAE (seconds):", round(weather_fold1_mae, 2))

Weather-inclusive Gradient Boosting — Fold 1
Training laps: 43482
Validation laps: 11659
MAE (seconds): 4.19


In [165]:
# Select the second race-grouped cross-validation fold
weather_train_idx, weather_val_idx = cv_splits[1]

# Separate the training and validation laps
X_weather_train = X_train_circuit.iloc[weather_train_idx].copy()
y_weather_train = y_train.iloc[weather_train_idx].copy()

X_weather_val = X_train_circuit.iloc[weather_val_idx].copy()
y_weather_val = y_train.iloc[weather_val_idx].copy()

# Keep only eligible laps in the training data
weather_train_eligible = eligible_training_laps.loc[X_weather_train.index]

X_weather_train = X_weather_train.loc[weather_train_eligible].copy()
y_weather_train = y_weather_train.loc[weather_train_eligible].copy()

# Create a fresh model with the same settings as Fold 1
weather_cv_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train the model on the second fold's training races
weather_cv_model.fit(X_weather_train, y_weather_train)

# Predict lap durations for the held-out validation races
weather_val_predictions = weather_cv_model.predict(X_weather_val)

# Calculate the prediction error for Fold 2
weather_fold2_mae = mean_absolute_error(
    y_weather_val,
    weather_val_predictions
)

# Display the results
print("Weather-inclusive Gradient Boosting — Fold 2")
print("Training laps:", len(X_weather_train))
print("Validation laps:", len(X_weather_val))
print("MAE (seconds):", round(weather_fold2_mae, 2))

Weather-inclusive Gradient Boosting — Fold 2
Training laps: 43948
Validation laps: 11678
MAE (seconds): 7.83


In [166]:
# Select the third race-grouped cross-validation fold
weather_train_idx, weather_val_idx = cv_splits[2]

# Separate training and validation data
X_weather_train = X_train_circuit.iloc[weather_train_idx].copy()
y_weather_train = y_train.iloc[weather_train_idx].copy()

X_weather_val = X_train_circuit.iloc[weather_val_idx].copy()
y_weather_val = y_train.iloc[weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
weather_train_eligible = eligible_training_laps.loc[X_weather_train.index]

X_weather_train = X_weather_train.loc[weather_train_eligible].copy()
y_weather_train = y_weather_train.loc[weather_train_eligible].copy()

# Create a fresh Gradient Boosting model with the same settings
weather_cv_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train the model using the third fold's eligible training laps
weather_cv_model.fit(X_weather_train, y_weather_train)

# Predict lap durations for the held-out races
weather_val_predictions = weather_cv_model.predict(X_weather_val)

# Calculate the Mean Absolute Error
weather_fold3_mae = mean_absolute_error(
    y_weather_val,
    weather_val_predictions
)

# Display the results
print("Weather-inclusive Gradient Boosting — Fold 3")
print("Training laps:", len(X_weather_train))
print("Validation laps:", len(X_weather_val))
print("MAE (seconds):", round(weather_fold3_mae, 2))

Weather-inclusive Gradient Boosting — Fold 3
Training laps: 43671
Validation laps: 11640
MAE (seconds): 8.13


In [167]:
# Select the fourth race-grouped cross-validation fold
weather_train_idx, weather_val_idx = cv_splits[3]

# Separate the training and validation data
X_weather_train = X_train_circuit.iloc[weather_train_idx].copy()
y_weather_train = y_train.iloc[weather_train_idx].copy()

X_weather_val = X_train_circuit.iloc[weather_val_idx].copy()
y_weather_val = y_train.iloc[weather_val_idx].copy()

# Keep only eligible laps in the training data
weather_train_eligible = eligible_training_laps.loc[X_weather_train.index]

X_weather_train = X_weather_train.loc[weather_train_eligible].copy()
y_weather_train = y_weather_train.loc[weather_train_eligible].copy()

# Create a fresh model using the same settings
weather_cv_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train the model on eligible training laps
weather_cv_model.fit(X_weather_train, y_weather_train)

# Predict lap durations for the held-out races
weather_val_predictions = weather_cv_model.predict(X_weather_val)

# Calculate the Mean Absolute Error
weather_fold4_mae = mean_absolute_error(
    y_weather_val,
    weather_val_predictions
)

# Display the Fold 4 results
print("Weather-inclusive Gradient Boosting — Fold 4")
print("Training laps:", len(X_weather_train))
print("Validation laps:", len(X_weather_val))
print("MAE (seconds):", round(weather_fold4_mae, 2))

Weather-inclusive Gradient Boosting — Fold 4
Training laps: 44576
Validation laps: 11677
MAE (seconds): 7.72


In [168]:
# Select the fifth and final race-grouped cross-validation fold
weather_train_idx, weather_val_idx = cv_splits[4]

# Separate training and validation data
X_weather_train = X_train_circuit.iloc[weather_train_idx].copy()
y_weather_train = y_train.iloc[weather_train_idx].copy()

X_weather_val = X_train_circuit.iloc[weather_val_idx].copy()
y_weather_val = y_train.iloc[weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
weather_train_eligible = eligible_training_laps.loc[X_weather_train.index]

X_weather_train = X_weather_train.loc[weather_train_eligible].copy()
y_weather_train = y_weather_train.loc[weather_train_eligible].copy()

# Create a fresh Gradient Boosting model with identical settings
weather_cv_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train the model using the eligible training laps
weather_cv_model.fit(X_weather_train, y_weather_train)

# Predict lap durations for the held-out races
weather_val_predictions = weather_cv_model.predict(X_weather_val)

# Calculate the final fold's Mean Absolute Error
weather_fold5_mae = mean_absolute_error(
    y_weather_val,
    weather_val_predictions
)

# Display the results
print("Weather-inclusive Gradient Boosting — Fold 5")
print("Training laps:", len(X_weather_train))
print("Validation laps:", len(X_weather_val))
print("MAE (seconds):", round(weather_fold5_mae, 2))

Weather-inclusive Gradient Boosting — Fold 5
Training laps: 43107
Validation laps: 12160
MAE (seconds): 8.32


In [169]:
# Collect the weather-inclusive MAE results from all five folds
weather_cv_scores = [
    weather_fold1_mae,
    weather_fold2_mae,
    weather_fold3_mae,
    weather_fold4_mae,
    weather_fold5_mae
]

# Add the weather-inclusive results to our existing model comparison
model_comparison["Gradient Boosting + Weather"] = weather_cv_scores

# Save the updated comparison for our final analysis notebook
model_comparison.to_csv("f1_model_cv_results.csv")

# Display the updated table and average errors
print(model_comparison.round(2))

print("\nAverage MAE by model:")
print(model_comparison.mean().round(2))

      Linear Regression  Gradient Boosting  Random Forest  \
Fold                                                        
1                  4.29               4.64           7.09   
2                  6.31               7.68           8.66   
3                  5.39               6.32           6.38   
4                  5.30               5.92           7.77   
5                  7.26               7.45           8.54   

      Gradient Boosting + Weather  
Fold                               
1                            4.19  
2                            7.83  
3                            8.13  
4                            7.72  
5                            8.32  

Average MAE by model:
Linear Regression              5.71
Gradient Boosting              6.40
Random Forest                  7.69
Gradient Boosting + Weather    7.24
dtype: float64


In [170]:
# Import Linear Regression and the tools needed for evaluation
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# Select the first race-grouped cross-validation fold
linear_weather_train_idx, linear_weather_val_idx = cv_splits[0]

# Separate the training and validation observations
X_linear_weather_train = X_train_circuit.iloc[linear_weather_train_idx].copy()
y_linear_weather_train = y_train.iloc[linear_weather_train_idx].copy()

X_linear_weather_val = X_train_circuit.iloc[linear_weather_val_idx].copy()
y_linear_weather_val = y_train.iloc[linear_weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
linear_weather_eligible = eligible_training_laps.loc[
    X_linear_weather_train.index
]

X_linear_weather_train = X_linear_weather_train.loc[
    linear_weather_eligible
].copy()

y_linear_weather_train = y_linear_weather_train.loc[
    linear_weather_eligible
].copy()

# Build a Linear Regression model using tyre, circuit and weather predictors
linear_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model on eligible training laps
linear_weather_model.fit(
    X_linear_weather_train,
    y_linear_weather_train
)

# Predict lap durations for the held-out races
linear_weather_predictions = linear_weather_model.predict(
    X_linear_weather_val
)

# Calculate prediction error
linear_weather_fold1_mae = mean_absolute_error(
    y_linear_weather_val,
    linear_weather_predictions
)

# Display the Fold 1 result
print("Weather-inclusive Linear Regression — Fold 1")
print("Training laps:", len(X_linear_weather_train))
print("Validation laps:", len(X_linear_weather_val))
print("MAE (seconds):", round(linear_weather_fold1_mae, 2))

Weather-inclusive Linear Regression — Fold 1
Training laps: 43482
Validation laps: 11659
MAE (seconds): 4.71


In [171]:
# Select the second race-grouped cross-validation fold
linear_weather_train_idx, linear_weather_val_idx = cv_splits[1]

# Separate the training and validation observations
X_linear_weather_train = X_train_circuit.iloc[linear_weather_train_idx].copy()
y_linear_weather_train = y_train.iloc[linear_weather_train_idx].copy()

X_linear_weather_val = X_train_circuit.iloc[linear_weather_val_idx].copy()
y_linear_weather_val = y_train.iloc[linear_weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
linear_weather_eligible = eligible_training_laps.loc[
    X_linear_weather_train.index
]

X_linear_weather_train = X_linear_weather_train.loc[
    linear_weather_eligible
].copy()

y_linear_weather_train = y_linear_weather_train.loc[
    linear_weather_eligible
].copy()

# Build a fresh Linear Regression model with weather predictors
linear_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model using eligible training laps
linear_weather_model.fit(
    X_linear_weather_train,
    y_linear_weather_train
)

# Predict lap durations for the held-out races
linear_weather_predictions = linear_weather_model.predict(
    X_linear_weather_val
)

# Calculate the Mean Absolute Error for Fold 2
linear_weather_fold2_mae = mean_absolute_error(
    y_linear_weather_val,
    linear_weather_predictions
)

# Display the Fold 2 results
print("Weather-inclusive Linear Regression — Fold 2")
print("Training laps:", len(X_linear_weather_train))
print("Validation laps:", len(X_linear_weather_val))
print("MAE (seconds):", round(linear_weather_fold2_mae, 2))

Weather-inclusive Linear Regression — Fold 2
Training laps: 43948
Validation laps: 11678
MAE (seconds): 6.38


In [172]:
# Select the third race-grouped cross-validation fold
linear_weather_train_idx, linear_weather_val_idx = cv_splits[2]

# Separate training and validation data
X_linear_weather_train = X_train_circuit.iloc[linear_weather_train_idx].copy()
y_linear_weather_train = y_train.iloc[linear_weather_train_idx].copy()

X_linear_weather_val = X_train_circuit.iloc[linear_weather_val_idx].copy()
y_linear_weather_val = y_train.iloc[linear_weather_val_idx].copy()

# Remove disrupted and unknown-status laps from training only
linear_weather_eligible = eligible_training_laps.loc[
    X_linear_weather_train.index
]

X_linear_weather_train = X_linear_weather_train.loc[
    linear_weather_eligible
].copy()

y_linear_weather_train = y_linear_weather_train.loc[
    linear_weather_eligible
].copy()

# Create a fresh Linear Regression model with weather predictors
linear_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model on the eligible training laps
linear_weather_model.fit(
    X_linear_weather_train,
    y_linear_weather_train
)

# Predict lap durations for the held-out races
linear_weather_predictions = linear_weather_model.predict(
    X_linear_weather_val
)

# Calculate the Mean Absolute Error for Fold 3
linear_weather_fold3_mae = mean_absolute_error(
    y_linear_weather_val,
    linear_weather_predictions
)

# Display the results
print("Weather-inclusive Linear Regression — Fold 3")
print("Training laps:", len(X_linear_weather_train))
print("Validation laps:", len(X_linear_weather_val))
print("MAE (seconds):", round(linear_weather_fold3_mae, 2))

Weather-inclusive Linear Regression — Fold 3
Training laps: 43671
Validation laps: 11640
MAE (seconds): 5.52


In [173]:
# Select the fourth race-grouped cross-validation fold
linear_weather_train_idx, linear_weather_val_idx = cv_splits[3]

# Separate the training and validation observations
X_linear_weather_train = X_train_circuit.iloc[linear_weather_train_idx].copy()
y_linear_weather_train = y_train.iloc[linear_weather_train_idx].copy()

X_linear_weather_val = X_train_circuit.iloc[linear_weather_val_idx].copy()
y_linear_weather_val = y_train.iloc[linear_weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
linear_weather_eligible = eligible_training_laps.loc[
    X_linear_weather_train.index
]

X_linear_weather_train = X_linear_weather_train.loc[
    linear_weather_eligible
].copy()

y_linear_weather_train = y_linear_weather_train.loc[
    linear_weather_eligible
].copy()

# Create a fresh Linear Regression model with weather predictors
linear_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model using eligible training laps
linear_weather_model.fit(
    X_linear_weather_train,
    y_linear_weather_train
)

# Predict lap durations for the held-out races
linear_weather_predictions = linear_weather_model.predict(
    X_linear_weather_val
)

# Calculate the Mean Absolute Error for Fold 4
linear_weather_fold4_mae = mean_absolute_error(
    y_linear_weather_val,
    linear_weather_predictions
)

# Display the results
print("Weather-inclusive Linear Regression — Fold 4")
print("Training laps:", len(X_linear_weather_train))
print("Validation laps:", len(X_linear_weather_val))
print("MAE (seconds):", round(linear_weather_fold4_mae, 2))

Weather-inclusive Linear Regression — Fold 4
Training laps: 44576
Validation laps: 11677
MAE (seconds): 5.89


In [174]:
# Select the fifth and final race-grouped cross-validation fold
linear_weather_train_idx, linear_weather_val_idx = cv_splits[4]

# Separate the training and validation observations
X_linear_weather_train = X_train_circuit.iloc[linear_weather_train_idx].copy()
y_linear_weather_train = y_train.iloc[linear_weather_train_idx].copy()

X_linear_weather_val = X_train_circuit.iloc[linear_weather_val_idx].copy()
y_linear_weather_val = y_train.iloc[linear_weather_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
linear_weather_eligible = eligible_training_laps.loc[
    X_linear_weather_train.index
]

X_linear_weather_train = X_linear_weather_train.loc[
    linear_weather_eligible
].copy()

y_linear_weather_train = y_linear_weather_train.loc[
    linear_weather_eligible
].copy()

# Create a fresh Linear Regression model with weather predictors
linear_weather_model = Pipeline([
    ("preprocessor", circuit_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model using eligible training laps
linear_weather_model.fit(
    X_linear_weather_train,
    y_linear_weather_train
)

# Predict lap durations for the held-out races
linear_weather_predictions = linear_weather_model.predict(
    X_linear_weather_val
)

# Calculate the Mean Absolute Error for Fold 5
linear_weather_fold5_mae = mean_absolute_error(
    y_linear_weather_val,
    linear_weather_predictions
)

# Display the results
print("Weather-inclusive Linear Regression — Fold 5")
print("Training laps:", len(X_linear_weather_train))
print("Validation laps:", len(X_linear_weather_val))
print("MAE (seconds):", round(linear_weather_fold5_mae, 2))

Weather-inclusive Linear Regression — Fold 5
Training laps: 43107
Validation laps: 12160
MAE (seconds): 8.19


In [175]:
# Collect the weather-inclusive Linear Regression results
linear_weather_cv_scores = [
    linear_weather_fold1_mae,
    linear_weather_fold2_mae,
    linear_weather_fold3_mae,
    linear_weather_fold4_mae,
    linear_weather_fold5_mae
]

# Add the new results to our existing comparison table
model_comparison["Linear Regression + Weather"] = linear_weather_cv_scores

# Save all five model configurations to our results CSV
model_comparison.to_csv("f1_model_cv_results.csv")

# Display the updated comparison
print(model_comparison.round(2))

# Calculate the average MAE using full-precision results
print("\nAverage MAE by model:")
print(model_comparison.mean().round(2))

      Linear Regression  Gradient Boosting  Random Forest  \
Fold                                                        
1                  4.29               4.64           7.09   
2                  6.31               7.68           8.66   
3                  5.39               6.32           6.38   
4                  5.30               5.92           7.77   
5                  7.26               7.45           8.54   

      Gradient Boosting + Weather  Linear Regression + Weather  
Fold                                                            
1                            4.19                         4.71  
2                            7.83                         6.38  
3                            8.13                         5.52  
4                            7.72                         5.89  
5                            8.32                         8.19  

Average MAE by model:
Linear Regression              5.71
Gradient Boosting              6.40
Random Forest         

In [176]:
# Select the first race-grouped cross-validation fold
diagnostic_train_idx, diagnostic_val_idx = cv_splits[0]

# Separate training and validation observations
X_diagnostic_train = X_train_circuit.iloc[diagnostic_train_idx].copy()
y_diagnostic_train = y_train.iloc[diagnostic_train_idx].copy()

X_diagnostic_val = X_train_circuit.iloc[diagnostic_val_idx].copy()
y_diagnostic_val = y_train.iloc[diagnostic_val_idx].copy()

# Exclude disrupted and unknown-status laps from training only
diagnostic_eligible = eligible_training_laps.loc[
    X_diagnostic_train.index
]

X_diagnostic_train = X_diagnostic_train.loc[
    diagnostic_eligible
].copy()

y_diagnostic_train = y_diagnostic_train.loc[
    diagnostic_eligible
].copy()

# Create the same tyre-and-circuit Linear Regression model
diagnostic_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("regressor", LinearRegression())
])

# Train the model using the eligible training laps
diagnostic_model.fit(
    X_diagnostic_train[tyre_features],
    y_diagnostic_train
)

# Predict lap durations for the held-out races
diagnostic_predictions = diagnostic_model.predict(
    X_diagnostic_val[tyre_features]
)

# Store the actual and predicted lap durations
fold1_diagnostics = pd.DataFrame({
    "actual_lap_duration": y_diagnostic_val,
    "predicted_lap_duration": diagnostic_predictions
})

# Calculate prediction errors in seconds
fold1_diagnostics["error_seconds"] = (
    fold1_diagnostics["predicted_lap_duration"]
    - fold1_diagnostics["actual_lap_duration"]
)

# Calculate the absolute size of each prediction error
fold1_diagnostics["absolute_error"] = (
    fold1_diagnostics["error_seconds"].abs()
)

# Check the number of predictions and their average error
print("Fold 1 predictions:", len(fold1_diagnostics))
print(
    "Fold 1 MAE:",
    round(fold1_diagnostics["absolute_error"].mean(), 2),
    "seconds"
)

# Preview the diagnostic results
display(fold1_diagnostics.head())

Fold 1 predictions: 11659
Fold 1 MAE: 4.29 seconds


,actual_lap_duration,predicted_lap_duration,error_seconds,absolute_error
1923,131.796,84.533438,-47.262562,47.262562
1924,130.343,83.909425,-46.433575,46.433575
1925,120.187,83.909425,-36.277575,36.277575
1926,116.843,84.533438,-32.309562,32.309562
1927,118.312,84.533438,-33.778562,33.778562


In [177]:
# Get the original dataset rows for the first five predictions
first_five_indices = fold1_diagnostics.head().index

# Select useful information about these laps
lap_details = df.loc[
    first_five_indices,
    [
        "session_key",
        "race_name",
        "lap_number",
        "compound",
        "tyre_age",
        "lap_duration"
    ]
].copy()

# Add the model's predictions and absolute errors
lap_details["predicted_duration"] = fold1_diagnostics.loc[
    first_five_indices,
    "predicted_lap_duration"
]

lap_details["absolute_error"] = fold1_diagnostics.loc[
    first_five_indices,
    "absolute_error"
]

# Display the details so we can investigate the large errors
display(lap_details)

,session_key,race_name,lap_number,compound,tyre_age,lap_duration,predicted_duration,absolute_error
1923,7787,Melbourne,1,MEDIUM,0.0,131.796,84.533438,47.262562
1924,7787,Melbourne,1,HARD,0.0,130.343,83.909425,46.433575
1925,7787,Melbourne,1,HARD,0.0,120.187,83.909425,36.277575
1926,7787,Melbourne,1,MEDIUM,0.0,116.843,84.533438,32.309562
1927,7787,Melbourne,1,MEDIUM,0.0,118.312,84.533438,33.778562


In [178]:
# Add each validation lap's lap number to our diagnostic results
fold1_diagnostics["lap_number"] = df.loc[
    fold1_diagnostics.index,
    "lap_number"
]

# Separate opening laps from all later laps
opening_laps = fold1_diagnostics[
    fold1_diagnostics["lap_number"] == 1
]

later_laps = fold1_diagnostics[
    fold1_diagnostics["lap_number"] > 1
]

# Compare the average absolute prediction errors
print("Fold 1 prediction errors by lap category")
print()

print("Opening laps:", len(opening_laps))
print(
    "Opening-lap MAE:",
    round(opening_laps["absolute_error"].mean(), 2),
    "seconds"
)

print()

print("Later laps:", len(later_laps))
print(
    "Later-lap MAE:",
    round(later_laps["absolute_error"].mean(), 2),
    "seconds"
)

Fold 1 prediction errors by lap category

Opening laps: 211
Opening-lap MAE: 11.62 seconds

Later laps: 11448
Later-lap MAE: 4.16 seconds


In [179]:
# Calculate the total absolute prediction error for Fold 1
total_absolute_error = fold1_diagnostics["absolute_error"].sum()

# Calculate the total error contributed by opening laps
opening_absolute_error = opening_laps["absolute_error"].sum()

# Calculate what percentage of observations are opening laps
opening_lap_percentage = (
    len(opening_laps) / len(fold1_diagnostics)
) * 100

# Calculate what percentage of total error comes from opening laps
opening_error_percentage = (
    opening_absolute_error / total_absolute_error
) * 100

# Display the results
print(
    "Opening laps as a percentage of validation laps:",
    round(opening_lap_percentage, 2),
    "%"
)

print(
    "Opening laps' share of total absolute error:",
    round(opening_error_percentage, 2),
    "%"
)

Opening laps as a percentage of validation laps: 1.81 %
Opening laps' share of total absolute error: 4.9 %


In [180]:
# Calculate the average signed error for opening laps
opening_mean_error = opening_laps["error_seconds"].mean()

# Calculate the average signed error for later laps
later_mean_error = later_laps["error_seconds"].mean()

# Calculate the percentage of opening laps that were underpredicted
opening_underpredicted = (
    opening_laps["error_seconds"] < 0
).mean() * 100

# Display the diagnostic results
print("Average signed prediction error")
print("Opening laps:", round(opening_mean_error, 2), "seconds")
print("Later laps:", round(later_mean_error, 2), "seconds")

print()

print(
    "Opening laps underpredicted:",
    round(opening_underpredicted, 2),
    "%"
)

Average signed prediction error
Opening laps: -11.23 seconds
Later laps: -0.88 seconds

Opening laps underpredicted: 91.47 %


### Diagnostic Analysis: Opening-Lap Prediction Errors

An initial diagnostic investigation was conducted using the validation predictions from the first fold of the five-fold race-grouped cross-validation experiment.

The tyre-and-circuit Linear Regression model generated **11,659 validation predictions**, achieving a Mean Absolute Error (MAE) of approximately **4.29 seconds**, consistent with the previously recorded cross-validation result.

Of these observations, **211 were opening laps**, representing approximately **1.81%** of the validation data. Opening laps had an MAE of **11.62 seconds**, compared with **4.16 seconds** for subsequent laps.

Although opening laps accounted for only 1.81% of validation observations, they contributed approximately **4.90% of the total absolute prediction error**.

Further analysis revealed an average signed prediction error of **−11.23 seconds** for opening laps, compared with **−0.88 seconds** for later laps. Approximately **91.47% of opening laps were underpredicted**, indicating a systematic tendency to predict shorter lap durations than those observed.

This pattern is consistent with the possibility that race-start conditions, including standing starts and traffic, are not adequately represented by the model's tyre compound, estimated tyre age, and circuit predictors.

However, these factors were not directly isolated or tested, and the analysis does not establish a causal explanation for the observed errors.

These findings are limited to the first cross-validation fold and should not yet be generalised to the complete dataset. Further diagnostic analysis across the remaining folds is required.

The opening laps were retained in the validation data, and the diagnostic investigation did not alter the reported model-performance results.

In [181]:
# Create an empty list to collect predictions from all five folds
all_fold_diagnostics = []

# Repeat the diagnostic analysis for each race-grouped fold
for fold_number, (fold_train_idx, fold_val_idx) in enumerate(
    cv_splits, start=1
):

    # Select this fold's training and validation observations
    fold_X_train = X_train_circuit.iloc[fold_train_idx].copy()
    fold_y_train = y_train.iloc[fold_train_idx].copy()

    fold_X_val = X_train_circuit.iloc[fold_val_idx].copy()
    fold_y_val = y_train.iloc[fold_val_idx].copy()

    # Remove disrupted and unknown-status laps from training only
    fold_eligible = eligible_training_laps.loc[
        fold_X_train.index
    ]

    fold_X_train = fold_X_train.loc[fold_eligible].copy()
    fold_y_train = fold_y_train.loc[fold_eligible].copy()

    # Create a fresh Linear Regression model for this fold
    fold_model = Pipeline([
        ("preprocessor", tyre_preprocessor),
        ("regressor", LinearRegression())
    ])

    # Train using eligible laps from the fold's training races
    fold_model.fit(
        fold_X_train[tyre_features],
        fold_y_train
    )

    # Predict laps from races held out in this fold
    fold_predictions = fold_model.predict(
        fold_X_val[tyre_features]
    )

    # Store actual values, predictions, and lap numbers
    fold_results = pd.DataFrame({
        "fold": fold_number,
        "actual_lap_duration": fold_y_val,
        "predicted_lap_duration": fold_predictions,
        "lap_number": df.loc[fold_y_val.index, "lap_number"]
    })

    # Calculate signed and absolute prediction errors
    fold_results["error_seconds"] = (
        fold_results["predicted_lap_duration"]
        - fold_results["actual_lap_duration"]
    )

    fold_results["absolute_error"] = (
        fold_results["error_seconds"].abs()
    )

    # Keep this fold's diagnostic results
    all_fold_diagnostics.append(fold_results)

# Combine all five folds into one diagnostic dataset
cv_diagnostics = pd.concat(all_fold_diagnostics)

# Check the number of predictions and overall MAE
print("Total validation predictions:", len(cv_diagnostics))
print("Number of folds:", cv_diagnostics["fold"].nunique())

print(
    "Overall lap-weighted MAE:",
    round(cv_diagnostics["absolute_error"].mean(), 2),
    "seconds"
)

# Preview the combined results
display(cv_diagnostics.head())

Total validation predictions: 58814
Number of folds: 5
Overall lap-weighted MAE: 5.72 seconds


,fold,actual_lap_duration,predicted_lap_duration,lap_number,error_seconds,absolute_error
1923,1,131.796,84.533438,1,-47.262562,47.262562
1924,1,130.343,83.909425,1,-46.433575,46.433575
1925,1,120.187,83.909425,1,-36.277575,36.277575
1926,1,116.843,84.533438,1,-32.309562,32.309562
1927,1,118.312,84.533438,1,-33.778562,33.778562


In [182]:
# Label each prediction as an opening lap or a later lap
cv_diagnostics["lap_category"] = np.where(
    cv_diagnostics["lap_number"] == 1,
    "Opening lap",
    "Later lap"
)

# Summarise prediction errors for each lap category
lap_category_summary = (
    cv_diagnostics
    .groupby("lap_category")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean"),
        underprediction_percentage=(
            "error_seconds",
            lambda errors: (errors < 0).mean() * 100
        )
    )
)

# Calculate each category's share of the validation observations
lap_category_summary["lap_percentage"] = (
    lap_category_summary["lap_count"]
    / len(cv_diagnostics)
    * 100
)

# Calculate each category's contribution to total absolute error
error_by_category = cv_diagnostics.groupby(
    "lap_category"
)["absolute_error"].sum()

lap_category_summary["total_error_percentage"] = (
    error_by_category
    / cv_diagnostics["absolute_error"].sum()
    * 100
)

# Display the combined diagnostic results
display(lap_category_summary.round(2))


,lap_count,mae_seconds,mean_signed_error,underprediction_percentage,lap_percentage,total_error_percentage
lap_category,,,,,,
Later lap,57815,5.55,-1.35,48.34,98.3,95.24
Opening lap,999,16.04,-15.71,94.19,1.7,4.76


### Diagnostic Analysis: Opening-Lap Errors Across Five Cross-Validation Folds

To investigate whether the opening-lap prediction errors observed in Fold 1 persisted across other races, out-of-fold predictions were collected from all five race-grouped cross-validation folds.

The analysis included **58,814 validation predictions**, with an overall lap-weighted Mean Absolute Error (MAE) of approximately **5.72 seconds**. This is consistent with the previously reported mean fold MAE of approximately 5.71 seconds, with the small difference arising from the averaging method.

Of these predictions, **999 observations were opening laps**, representing approximately **1.70%** of the validation dataset. The remaining 57,815 observations were from later laps.

Opening laps recorded an MAE of **16.04 seconds**, compared with **5.55 seconds** for later laps. Opening-lap errors were therefore approximately **2.9 times larger** on average.

The mean signed prediction error for opening laps was **−15.71 seconds**, indicating that predicted lap durations were substantially shorter than observed durations on average. Approximately **94.19% of opening laps were underpredicted**.

In comparison, later laps had a mean signed prediction error of **−1.35 seconds**, with approximately **48.34%** of predictions falling below the observed lap duration.

Although opening laps represented only 1.70% of validation observations, they contributed approximately **4.76% of total absolute prediction error**.

These findings indicate a systematic tendency for the tyre-and-circuit Linear Regression model to underestimate opening-lap duration across the evaluated race groups.

Race-start conditions, including standing starts and traffic, are plausible contributing factors because they are not explicitly represented among the model's predictors. However, this analysis does not isolate those effects or establish a causal explanation.

The results provide evidence of an important model limitation. All opening laps were retained in the validation data, and the diagnostic analysis did not alter the reported cross-validation performance.

These findings relate to development-stage cross-validation rather than a fully independent final test.

In [183]:
# Add each validation lap's tyre compound to our diagnostic results
cv_diagnostics["compound"] = df.loc[
    cv_diagnostics.index,
    "compound"
]

# Calculate prediction accuracy separately for each tyre compound
compound_diagnostics = (
    cv_diagnostics
    .groupby("compound")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean")
    )
)

# Calculate the percentage of predictions that were too fast
compound_diagnostics["underprediction_percentage"] = (
    cv_diagnostics
    .groupby("compound")["error_seconds"]
    .apply(lambda errors: (errors < 0).mean() * 100)
)

# Sort compounds from lowest to highest prediction error
compound_diagnostics = compound_diagnostics.sort_values(
    "mae_seconds"
)

# Display the results
display(compound_diagnostics.round(2))

,lap_count,mae_seconds,mean_signed_error,underprediction_percentage
compound,,,,
SOFT,6188,4.83,-2.17,49.03
HARD,27186,5.30,-1.34,53.65
MEDIUM,22118,5.97,-1.13,44.66
INTERMEDIATE,3294,9.10,-5.71,41.89
WET,28,22.30,-12.45,46.43


In [184]:
# Select later laps only for this diagnostic comparison
later_lap_diagnostics = cv_diagnostics.loc[
    cv_diagnostics["lap_number"] > 1
].copy()

# Calculate tyre-compound errors without opening laps
later_compound_summary = (
    later_lap_diagnostics
    .groupby("compound")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean")
    )
)

# Sort compounds from lowest to highest average error
later_compound_summary = later_compound_summary.sort_values(
    "mae_seconds"
)

# Display the diagnostic comparison
display(later_compound_summary.round(2))

,lap_count,mae_seconds,mean_signed_error
compound,,,
SOFT,6001,4.41,-1.67
HARD,27063,5.25,-1.27
MEDIUM,21496,5.76,-0.81
INTERMEDIATE,3227,8.51,-5.05
WET,28,22.30,-12.45


In [185]:
# Select the wet-tyre validation predictions
wet_diagnostics = cv_diagnostics.loc[
    cv_diagnostics["compound"] == "WET"
].copy()

# Add the race identifiers and names from the original dataset
wet_diagnostics["session_key"] = df.loc[
    wet_diagnostics.index,
    "session_key"
]

wet_diagnostics["race_name"] = df.loc[
    wet_diagnostics.index,
    "race_name"
]

# Count wet-tyre laps in each race
wet_race_summary = (
    wet_diagnostics
    .groupby(["session_key", "race_name"])
    .agg(
        wet_laps=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean")
    )
    .sort_values("wet_laps", ascending=False)
)

# Display how widely the wet-tyre observations are distributed
print("Total wet-tyre laps:", len(wet_diagnostics))
print("Races containing wet-tyre laps:", len(wet_race_summary))

display(wet_race_summary.round(2))

Total wet-tyre laps: 28
Races containing wet-tyre laps: 2


,,wet_laps,mae_seconds
session_key,race_name,,
9531,Canada,15,9.20
9636,Brazil,13,37.42


In [186]:
# Select wet-tyre predictions from the Brazilian Grand Prix
brazil_wet_laps = wet_diagnostics.loc[
    wet_diagnostics["session_key"] == 9636
].copy()

# Show the lap numbers and prediction errors
brazil_wet_details = brazil_wet_laps[
    [
        "fold",
        "lap_number",
        "actual_lap_duration",
        "predicted_lap_duration",
        "error_seconds",
        "absolute_error"
    ]
].sort_values("lap_number")

# Display all 13 wet-tyre observations
display(brazil_wet_details.round(2))

,fold,lap_number,actual_lap_duration,predicted_lap_duration,error_seconds,absolute_error
45736,4,29,101.66,93.49,-8.17,8.17
45761,4,29,127.54,93.80,-33.74,33.74
45743,4,30,114.80,93.80,-21.00,21.00
45748,4,30,114.92,92.87,-22.05,22.05
45751,4,30,128.47,93.43,-35.05,35.05
45756,4,30,125.36,93.37,-32.00,32.00
45760,4,31,125.20,93.74,-31.46,31.46
45765,4,31,112.06,92.81,-19.25,19.25
45769,4,31,111.02,93.37,-17.65,17.65
45773,4,31,160.52,93.30,-67.22,67.22


In [187]:
# Select the Brazilian Grand Prix session
brazil_session = 9636

# Find safety-car and virtual-safety-car intervals for this race
brazil_sc_vsc = all_disruption_intervals.loc[
    all_disruption_intervals["session_key"] == brazil_session
].copy()

# Find red-flag intervals for this race
brazil_red_flags = all_red_flag_intervals.loc[
    all_red_flag_intervals["session_key"] == brazil_session
].copy()

# Display the recorded disruption intervals
print("Brazil: Safety Car / Virtual Safety Car intervals")
display(brazil_sc_vsc)

print("Brazil: Red Flag intervals")
display(brazil_red_flags)

Brazil: Safety Car / Virtual Safety Car intervals


,session_key,type,start,end
31,9636,VSC,2024-11-03 16:28:21+00:00,2024-11-03 16:29:50+00:00
32,9636,SC,2024-11-03 16:32:53+00:00,2024-11-03 16:37:22+00:00
33,9636,SC,2024-11-03 17:13:14+00:00,2024-11-03 17:18:04+00:00


Brazil: Red Flag intervals


,session_key,start,end,duration_minutes
7,9636,2024-11-03 16:37:22+00:00,2024-11-03 17:02:00.390000+00:00,24.639833


In [188]:
# Get the original row indices for the 13 Brazilian wet-tyre laps
brazil_wet_indices = brazil_wet_laps.index

# Retrieve the previously calculated race-control flags
brazil_disruption_check = all_training_laps.loc[
    brazil_wet_indices,
    [
        "lap_number",
        "sc_vsc_overlap",
        "red_flag_overlap",
        "race_control_disruption"
    ]
].copy()

# Add each lap's absolute prediction error
brazil_disruption_check["absolute_error"] = (
    brazil_wet_laps.loc[
        brazil_wet_indices,
        "absolute_error"
    ]
)

# Sort the results by lap number
brazil_disruption_check = brazil_disruption_check.sort_values(
    "lap_number"
)

# Display the individual lap flags
display(brazil_disruption_check)

# Count how many wet-tyre laps overlapped a recorded disruption
print(
    "Wet-tyre laps overlapping race-control disruptions:",
    brazil_disruption_check["race_control_disruption"].sum(),
    "out of",
    len(brazil_disruption_check)
)

,lap_number,sc_vsc_overlap,red_flag_overlap,race_control_disruption,absolute_error
45736,29,False,False,False,8.172568
45761,29,True,False,True,33.744377
45743,30,True,False,True,20.999377
45748,30,True,False,True,22.048951
45751,30,True,False,True,35.045207
45756,30,True,False,True,31.996845
45760,31,True,False,True,31.462015
45765,31,True,False,True,19.249589
45769,31,True,False,True,17.648845
45773,31,True,True,True,67.215483


Wet-tyre laps overlapping race-control disruptions: 12 out of 13


In [189]:
# Compare wet-tyre errors by recorded race-control disruption status
brazil_disruption_summary = (
    brazil_disruption_check
    .groupby("race_control_disruption")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean")
    )
)

# Give the rows clearer labels
brazil_disruption_summary.index = (
    brazil_disruption_summary.index.map({
        False: "No recorded overlap",
        True: "Recorded disruption overlap"
    })
)

# Display the comparison
display(brazil_disruption_summary.round(2))

,lap_count,mae_seconds
race_control_disruption,,
No recorded overlap,1,8.17
Recorded disruption overlap,12,39.86


### Diagnostic Analysis: Wet-Tyre Errors and Race-Control Disruptions

The five-fold cross-validation diagnostics revealed that the Linear Regression model recorded a substantially higher Mean Absolute Error (MAE) for wet-tyre observations than for dry-weather compounds.

However, the wet-tyre validation sample contained only **28 laps across two races**: 15 laps from Canada and 13 from Brazil. The corresponding race-level MAEs were approximately **9.20 seconds** and **37.42 seconds**, respectively.

Further investigation of the Brazilian Grand Prix observations revealed that all 13 wet-tyre laps occurred between race laps 29 and 32.

Using the previously constructed race-control disruption flags, **12 of the 13 Brazilian wet-tyre laps (92.3%)** were identified as overlapping a recorded safety-car, virtual-safety-car, or red-flag interval.

The 12 disruption-overlapping observations had an MAE of approximately **39.86 seconds**, compared with **8.17 seconds** for the single observation without a recorded disruption overlap.

These findings indicate that the unusually high wet-tyre prediction error recorded for Brazil was strongly associated with race-control disruptions. The model was trained using laps without identified race-control disruptions, whereas validation observations were deliberately retained regardless of disruption status.

Consequently, the wet-tyre MAE reflects not only potential difficulties in modelling wet-weather performance but also the influence of unusual race conditions that were not explicitly represented among the model's predictors.

Importantly, the presence of only one Brazilian wet-tyre lap without a recorded disruption overlap prevents a reliable assessment of model performance during uninterrupted wet-weather racing at that event.

The findings therefore **do not establish that wet tyres themselves caused the larger prediction errors**, nor do they demonstrate that the model would perform accurately on wet tyres in the absence of disruptions.

The diagnostic investigation did not alter the original cross-validation results. Instead, it highlights the importance of considering race-control conditions and sample size when interpreting model performance by tyre compound.

In [190]:
# Select wet-tyre validation laps from the Canadian Grand Prix
canada_wet_laps = wet_diagnostics.loc[
    wet_diagnostics["session_key"] == 9531
].copy()

# Retrieve the race-control flags calculated earlier
canada_disruption_check = all_training_laps.loc[
    canada_wet_laps.index,
    [
        "lap_number",
        "sc_vsc_overlap",
        "red_flag_overlap",
        "race_control_disruption"
    ]
].copy()

# Add the absolute prediction error for each lap
canada_disruption_check["absolute_error"] = (
    canada_wet_laps["absolute_error"]
)

# Display the laps in race-lap order
display(
    canada_disruption_check
    .sort_values("lap_number")
    .round(2)
)

# Count how many laps overlapped recorded disruptions
print(
    "Canadian wet-tyre laps overlapping disruptions:",
    canada_disruption_check["race_control_disruption"].sum(),
    "out of",
    len(canada_disruption_check)
)

,lap_number,sc_vsc_overlap,red_flag_overlap,race_control_disruption,absolute_error
31672,2,False,False,False,8.10
31677,2,False,False,False,6.25
31689,3,False,False,False,8.71
31694,3,False,False,False,8.19
31708,4,False,False,False,10.80
31712,4,False,False,False,9.74
31728,5,False,False,False,10.22
31732,5,False,False,False,8.64
31748,6,False,False,False,10.02
31752,6,False,False,False,8.52


Canadian wet-tyre laps overlapping disruptions: 0 out of 15


### Diagnostic Analysis: Comparison of Wet-Tyre Predictions in Canada and Brazil

The wet-tyre diagnostic investigation was extended to examine the distribution of recorded race-control disruptions across both races containing wet-tyre validation observations.

The Canadian Grand Prix contributed **15 wet-tyre laps**, covering race laps 2–11. None of these observations overlapped the safety-car, virtual-safety-car, or red-flag intervals identified by the race-control analysis.

The Canadian wet-tyre observations recorded a Mean Absolute Error (MAE) of approximately **9.20 seconds**.

In contrast, the Brazilian Grand Prix contributed **13 wet-tyre laps**, of which **12 (92.3%)** overlapped recorded race-control disruptions. The Brazilian wet-tyre observations recorded a substantially higher MAE of approximately **37.42 seconds**.

The disruption-overlapping Brazilian observations had an MAE of approximately **39.86 seconds**, while the single Brazilian observation without a recorded overlap had an absolute error of approximately **8.17 seconds**.

These findings indicate that the exceptionally high wet-tyre prediction error in Brazil was strongly associated with race-control disruptions. Canada's lower error demonstrates that the two wet-tyre race samples were evaluated under different recorded disruption conditions.

Nevertheless, Canada's MAE remained higher than the errors observed for the three dry-weather tyre compounds. This suggests that predicting wet-tyre lap durations may remain challenging even when no recorded race-control disruption overlap is identified.

The evidence remains limited because the entire wet-tyre validation sample comprised only **28 laps from two races**. Furthermore, an absence of recorded disruption overlap does not establish that all other racing conditions were normal or comparable.

Therefore, the results cannot isolate the independent effect of wet tyres on predictive accuracy, nor can they establish the model's general performance across wet-weather Formula 1 races.

The investigation highlights the importance of interpreting tyre-compound prediction errors alongside race-control conditions, sample sizes, and race-specific circumstances.

These analyses were diagnostic only and did not modify the original cross-validation results.

In [191]:
# Add the circuit information used in our model
cv_diagnostics["circuit"] = X_train_circuit.loc[
    cv_diagnostics.index,
    "circuit"
]

# Calculate prediction errors for each circuit
circuit_diagnostics = (
    cv_diagnostics
    .groupby("circuit")
    .agg(
        lap_count=("absolute_error", "size"),
        race_count=("fold", "nunique"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean")
    )
)

# Sort circuits from highest to lowest average prediction error
circuit_diagnostics = circuit_diagnostics.sort_values(
    "mae_seconds",
    ascending=False
)

# Display the circuit comparison
display(circuit_diagnostics.round(2))

,lap_count,race_count,mae_seconds,mean_signed_error
circuit,,,,
Spielberg,2380,1,18.96,17.82
Las Vegas,912,1,17.74,-17.74
Imola,1168,1,10.57,3.72
Suzuka,2724,1,9.65,-9.65
Sakhir,1005,1,9.16,-9.16
Shanghai,1040,1,9.11,-9.11
Silverstone,1701,2,7.82,-5.21
Budapest,1339,1,7.46,7.38
Melbourne,2690,3,7.42,-5.97


In [192]:
# Add the original race session identifier to each validation prediction
cv_diagnostics["session_key"] = df.loc[
    cv_diagnostics.index,
    "session_key"
]

# Count distinct race sessions at each circuit
actual_race_counts = (
    cv_diagnostics
    .groupby("circuit")["session_key"]
    .nunique()
)

# Replace the fold count with the correct number of races
circuit_diagnostics["race_count"] = actual_race_counts

# Display the corrected circuit comparison
display(circuit_diagnostics.round(2))

,lap_count,race_count,mae_seconds,mean_signed_error
circuit,,,,
Spielberg,2380,2,18.96,17.82
Las Vegas,912,1,17.74,-17.74
Imola,1168,1,10.57,3.72
Suzuka,2724,3,9.65,-9.65
Sakhir,1005,1,9.16,-9.16
Shanghai,1040,1,9.11,-9.11
Silverstone,1701,2,7.82,-5.21
Budapest,1339,1,7.46,7.38
Melbourne,2690,3,7.42,-5.97


In [194]:
# Select validation predictions from Spielberg
spielberg_diagnostics = cv_diagnostics.loc[
    cv_diagnostics["circuit"] == "Spielberg"
].copy()

# Calculate prediction errors separately for each race session
# The session_key uniquely identifies each race
spielberg_race_summary = (
    spielberg_diagnostics
    .groupby("session_key")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean")
    )
)

# Display the two Spielberg races separately
display(spielberg_race_summary.round(2))

,lap_count,mae_seconds,mean_signed_error
session_key,,,
9118,1288,18.92,18.02
9955,1092,19.01,17.58


In [195]:
# Compare actual and predicted lap durations for each Spielberg race
spielberg_pace_summary = (
    spielberg_diagnostics
    .groupby("session_key")
    .agg(
        lap_count=("actual_lap_duration", "size"),
        actual_mean=("actual_lap_duration", "mean"),
        predicted_mean=("predicted_lap_duration", "mean"),
        actual_median=("actual_lap_duration", "median"),
        predicted_median=("predicted_lap_duration", "median")
    )
)

# Calculate the difference between average predicted and actual lap times
spielberg_pace_summary["mean_difference"] = (
    spielberg_pace_summary["predicted_mean"]
    - spielberg_pace_summary["actual_mean"]
)

# Display the results in seconds
display(spielberg_pace_summary.round(2))

,lap_count,actual_mean,predicted_mean,actual_median,predicted_median,mean_difference
session_key,,,,,,
9118,1288,71.97,89.99,70.58,90.03,18.02
9955,1092,72.49,90.08,70.80,90.17,17.58


In [196]:
# Check whether Spielberg appeared in each fold's training data
for fold_number, (fold_train_idx, fold_val_idx) in enumerate(
    cv_splits, start=1
):

    # Retrieve the circuit names in this fold
    training_circuits = X_train_circuit.iloc[
        fold_train_idx
    ]["circuit"]

    validation_circuits = X_train_circuit.iloc[
        fold_val_idx
    ]["circuit"]

    # Only report folds containing Spielberg validation laps
    if (validation_circuits == "Spielberg").any():

        # Count distinct Spielberg races in training and validation
        training_races = df.loc[
            training_circuits.index[training_circuits == "Spielberg"],
            "session_key"
        ].nunique()

        validation_races = df.loc[
            validation_circuits.index[validation_circuits == "Spielberg"],
            "session_key"
        ].nunique()

        print("Fold:", fold_number)
        print("Spielberg training races:", training_races)
        print("Spielberg validation races:", validation_races)
        print()

Fold: 5
Spielberg training races: 0
Spielberg validation races: 2



In [197]:
# Identify circuits appearing in validation but not in training
for fold_number, (fold_train_idx, fold_val_idx) in enumerate(
    cv_splits, start=1
):

    # Get the unique circuits in the training and validation sets
    training_circuits = set(
        X_train_circuit.iloc[fold_train_idx]["circuit"]
    )

    validation_circuits = set(
        X_train_circuit.iloc[fold_val_idx]["circuit"]
    )

    # Find circuits the model has never encountered during training
    unseen_circuits = sorted(
        validation_circuits - training_circuits
    )

    # Display the results for this fold
    print(f"Fold {fold_number}: {unseen_circuits}")

Fold 1: ['Shanghai']
Fold 2: ['Las Vegas']
Fold 3: ['Budapest', 'Imola']
Fold 4: ['Suzuka']
Fold 5: ['Miami Gardens', 'Sakhir', 'Spielberg']


In [198]:
# Start by marking every validation prediction as a seen circuit
cv_diagnostics["circuit_status"] = "Seen circuit"

# Examine the training and validation circuits in each fold
for fold_number, (fold_train_idx, fold_val_idx) in enumerate(
    cv_splits, start=1
):

    # Find the circuits represented in this fold's training data
    training_circuits = set(
        X_train_circuit.iloc[fold_train_idx]["circuit"]
    )

    # Identify validation laps from circuits absent during training
    validation_data = X_train_circuit.iloc[fold_val_idx]

    unseen_lap_indices = validation_data.index[
        ~validation_data["circuit"].isin(training_circuits)
    ]

    # Mark those validation predictions as unseen circuits
    cv_diagnostics.loc[
        unseen_lap_indices, "circuit_status"
    ] = "Unseen circuit"

# Compare prediction accuracy for seen and unseen circuits
circuit_status_summary = (
    cv_diagnostics
    .groupby("circuit_status")
    .agg(
        lap_count=("absolute_error", "size"),
        mae_seconds=("absolute_error", "mean"),
        mean_signed_error=("error_seconds", "mean")
    )
)

# Display the comparison
display(circuit_status_summary.round(2))

,lap_count,mae_seconds,mean_signed_error
circuit_status,,,
Seen circuit,47723,4.32,-1.83
Unseen circuit,11091,11.78,-0.59


### Diagnostic Analysis: Generalisation to Seen and Unseen Circuits

The five-fold race-grouped cross-validation results were examined to assess whether prediction accuracy differed between circuits represented in the training data and circuits absent from training.

The Linear Regression model achieved a mean absolute error (MAE) of **4.32 seconds across 47,723 validation laps at previously seen circuits**, compared with **11.78 seconds across 11,091 validation laps at unseen circuits**. The error at unseen circuits was approximately 2.73 times higher.

This difference helps explain the unusually high prediction errors observed at certain circuits, particularly Spielberg and Las Vegas. In the Spielberg validation fold, neither of the two Spielberg races was represented in the training data. Both races showed substantial overprediction of lap duration.

The model uses one-hot encoding for circuit identity, with unknown categories ignored during transformation. Consequently, when a circuit is absent from training, the model cannot apply a circuit-specific adjustment learned from previous races at that location.

These findings indicate that the model's predictive performance is considerably weaker when generalising to unfamiliar circuits. However, the comparison is descriptive rather than causal, as the seen and unseen groups comprise different circuits, races and conditions. Other factors may also contribute to the observed error differences.

The official five-fold cross-validation scores remain unchanged. This analysis provides additional context for interpreting the model's generalisation performance and its limitations.

In [199]:
# Save all five folds of Linear Regression validation predictions
# This includes lap errors, tyre compounds, circuits, and race identifiers
cv_diagnostics.to_csv(
    "f1_linear_cv_diagnostics.csv",
    index=False
)

# Confirm that the file was saved and show its dimensions
print("Saved: f1_linear_cv_diagnostics.csv")
print("Rows:", len(cv_diagnostics))
print("Columns:", len(cv_diagnostics.columns))

Saved: f1_linear_cv_diagnostics.csv
Rows: 58814
Columns: 11


In [200]:

# Import the tools needed to build and save our prediction model
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import joblib

# Build a complete prediction pipeline using our existing
# tyre-and-circuit preprocessing configuration
lap_prediction_model = Pipeline([
    ("preprocessor", tyre_preprocessor),
    ("model", LinearRegression())
])

# Train the model using only our original training races
# This avoids fitting the model on the held-out test races
lap_prediction_model.fit(
    X_train_circuit[tyre_features],
    y.iloc[train_idx]
)

# Save the trained model to a file
joblib.dump(
    lap_prediction_model,
    "lapd_prediction_model.joblib"
)

# Confirm that the model was saved
print("🏁 LAPD prediction model saved successfully!")


🏁 LAPD prediction model saved successfully!


In [201]:

# Display the exact circuit names used to train our model
print(sorted(X_train_circuit["circuit"].dropna().unique()))


['Austin', 'Baku', 'Barcelona', 'Budapest', 'Imola', 'Jeddah', 'Las Vegas', 'Lusail', 'Marina Bay', 'Melbourne', 'Mexico City', 'Miami Gardens', 'Monaco', 'Montréal', 'Monza', 'Sakhir', 'Shanghai', 'Silverstone', 'Spa-Francorchamps', 'Spielberg', 'Suzuka', 'São Paulo', 'Yas Island', 'Zandvoort']


In [202]:

# Get the tyre compound effects learned by our saved model
# These values show how each compound changes the predicted lap time
feature_names = (
    lap_prediction_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get the coefficients from the Linear Regression model
coefficients = (
    lap_prediction_model
    .named_steps["model"]
    .coef_
)

# Display only the coefficients related to tyre compounds
for feature, coefficient in zip(feature_names, coefficients):
    if "compound" in feature:
        print(f"{feature}: {coefficient:.2f} seconds")


categories__compound_HARD: -9.76 seconds
categories__compound_INTERMEDIATE: 5.04 seconds
categories__compound_MEDIUM: -8.84 seconds
categories__compound_SOFT: -8.89 seconds
categories__compound_WET: 22.45 seconds
